# 05+FE — Feature Engineering & Benchmark Multi-Algorithmes
## IndabaX Cameroon 2026 — Prédiction Qualité de l'Air

**Pipeline complet** : Feature Engineering → Benchmark → Modèle Final  
**Algorithmes** : LightGBM · XGBoost · CatBoost · Random Forest · Extra Trees  
**Cibles** : pm2_5, pm10, no2, o3, co, so2, aqi_global  
**Objectif** : Feature engineering intégré + Benchmark optimisé, prêt pour Kaggle sans renommage de chemins

## Section 0 — Imports & Configuration

In [1]:
import os, sys, time, warnings, pathlib, json, logging
from datetime import datetime
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False

# ── Logger ─────────────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger(__name__)

# ── Affichage ──────────────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (12, 5)})
sns.set_style('whitegrid')

print('Imports OK')

Imports OK


In [2]:
# ── Chemins — Structure locale (notebook/ data/ model/ au même niveau) ────
# Structure attendue :
#   projet/
#     data/        ← données brutes (Base_complete.parquet ou .xlsx)
#     models/       ← sorties : modèles PKL, figures, registre JSON
#     notebook/    ← ce fichier
#
# ROOT pointe sur le dossier du notebook (= '.' quand Jupyter démarre ici).
# On remonte d'un niveau avec .parent pour atteindre le dossier projet.

ROOT       = pathlib.Path('.').resolve()
KAGGLE_WD  = pathlib.Path('/kaggle/working')
KAGGLE_IN  = pathlib.Path('/kaggle/input/datasets/shun004/new-base')
IS_KAGGLE  = KAGGLE_WD.exists()

if IS_KAGGLE:
    DATA_IN  = KAGGLE_IN
    DATA_OUT = KAGGLE_WD
else:
    # ROOT = notebook/, donc ROOT.parent = projet/
    BASE_DIR = ROOT.parent
    DATA_IN  = BASE_DIR / 'data'    # lecture des données
    DATA_OUT = BASE_DIR / 'models'   # écriture des résultats

# Alias pour les sections Feature Engineering
DATA       = DATA_IN
DATA_WRITE = DATA_OUT   # écriture des fichiers produits

# Créer les dossiers de sortie si nécessaire
DATA_OUT.mkdir(parents=True, exist_ok=True)
MODELS_DIR = DATA_OUT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

log.info('IS_KAGGLE  : %s', IS_KAGGLE)
log.info('DATA_IN    : %s', DATA_IN)
log.info('DATA_OUT   : %s', DATA_OUT)
log.info('MODELS_DIR : %s', MODELS_DIR)

20:19:25 [INFO] IS_KAGGLE  : False
20:19:25 [INFO] DATA_IN    : C:\Users\NICK-TECH\OneDrive\Desktop\Deepstsat\data
20:19:25 [INFO] DATA_OUT   : C:\Users\NICK-TECH\OneDrive\Desktop\Deepstsat\models
20:19:25 [INFO] MODELS_DIR : C:\Users\NICK-TECH\OneDrive\Desktop\Deepstsat\models\models


In [3]:
# ── Constantes globales ────────────────────────────────────────────────────────
TARGETS      = ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'aqi_global']
TARGETS_LOG  = ['pm2_5', 'pm10', 'no2', 'co', 'so2']
RANDOM_STATE = 42
N_FOLDS      = 5
PATIENCE     = 50
LOG_EVERY    = 200

TRAIN_END    = '2024-09-30'
VALID_START  = '2024-10-01'
VALID_END    = '2024-12-31'
TEST_START   = '2025-01-01'

MAPE_THRESHOLDS = {
    'pm2_5': 1.0, 'pm10': 2.0, 'no2': 0.5,
    'o3': 5.0, 'co': 50.0, 'so2': 0.1, 'aqi_global': 5.0
}

# Résultats de référence NB06 ──────────────────────────────────────────────────
NB06_RMSE = {'pm2_5':1.56,'pm10':6.98,'no2':2.80,'o3':8.31,
             'co':24.9,'so2':0.51,'aqi_global':5.30}
OBJ_RMSE  = {'pm2_5':1.40,'pm10':6.50,'no2':2.50,'o3':7.80,
             'co':23.00,'so2':0.48,'aqi_global':4.80}

# Storage global ───────────────────────────────────────────────────────────────
results_dict    = {}   # {algo: {target: metrics_dict}}
all_test_preds  = {}   # {algo: {target: np.array}} en échelle originale
all_valid_preds = {}   # {algo: {target: np.array}} en échelle originale
model_registry  = {}   # {target: {algo: {...}}}
for t in TARGETS:
    model_registry[t] = {}

log.info('Configuration OK — %d cibles', len(TARGETS))


20:19:25 [INFO] Configuration OK — 7 cibles


---
## Section FE — Feature Engineering (intégré)

> Transforme le dataset brut en features enrichies.  
> Le `df` produit ici est directement utilisé par les sections de modélisation ci-dessous.  
> Les fichiers intermédiaires sont sauvegardés dans `DATA_WRITE` (`/kaggle/working` sur Kaggle, `data/` en local).

In [4]:
# ── FE 0.4  Chargement avec checkpoint ────────────────────────────────────────

def _find_file(root, *names):
    """Cherche un fichier par nom dans root et ses sous-dossiers (1 niveau)."""
    root = pathlib.Path(root)
    for name in names:
        if (root / name).exists():
            return root / name
        for sub in sorted(root.iterdir()):
            if sub.is_dir() and (sub / name).exists():
                return sub / name
    return None

# Cache dans /kaggle/working pour accélerer les relances
CKPT_WORK = DATA_WRITE / 'base_complete.parquet'

if CKPT_WORK.exists():
    log.info('[SKIP] Cache trouvé : %s', CKPT_WORK)
    df = pd.read_parquet(CKPT_WORK)
else:
    # 1) Chercher le parquet dans DATA_IN (et ses sous-dossiers)
    found_parquet = _find_file(DATA_IN, 'base_complete.parquet', 'Base_complete.parquet')
    # 2) Fallback : chercher le fichier Excel
    found_excel   = _find_file(DATA_IN, 'Base_complete.xlsx', 'base_complete.xlsx')

    if found_parquet:
        log.info('Parquet trouvé : %s', found_parquet)
        df = pd.read_parquet(found_parquet)
        df['time'] = pd.to_datetime(df['time'])
        df = df.sort_values(['city_indabax', 'time']).reset_index(drop=True)
        df.to_parquet(CKPT_WORK, index=False)
        log.info('Mis en cache → %s', CKPT_WORK)
    elif found_excel:
        log.info('Excel trouvé : %s — conversion en parquet…', found_excel)
        df = pd.read_excel(found_excel)
        df['time'] = pd.to_datetime(df['time'])
        df = df.sort_values(['city_indabax', 'time']).reset_index(drop=True)
        df.to_parquet(CKPT_WORK, index=False)
        log.info('Converti et mis en cache → %s', CKPT_WORK)
    else:
        # Afficher le contenu du dossier pour aider au diagnostic
        contents = list(DATA_IN.rglob('*'))[:30]
        raise FileNotFoundError(
            f"Ni base_complete.parquet ni Base_complete.xlsx trouvé dans {DATA_IN}\n"
            f"Contenu visible : {[str(p) for p in contents]}"
        )

log.info('Dataset chargé : %d lignes × %d colonnes', *df.shape)

20:19:25 [INFO] [SKIP] Cache trouvé : C:\Users\NICK-TECH\OneDrive\Desktop\Deepstsat\models\base_complete.parquet
20:19:26 [INFO] Dataset chargé : 87240 lignes × 32 colonnes


In [5]:
# ── 0.5  Aperçu rapide ───────────────────────────────────────────────────────
TARGETS = ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'aqi_global']

print('=== Shape :', df.shape)
print('\n=== Plage de dates :', df['time'].min().date(), '→', df['time'].max().date())
print('=== Nb villes      :', df['city_indabax'].nunique())
print('=== Nb régions     :', df['region_indabax'].nunique())
print('=== Valeurs manquantes totales :', df.isnull().sum().sum())
print('\n=== Types :')
print(df.dtypes)
print('\n=== Statistiques des cibles :')
df[TARGETS].describe().round(2)

=== Shape : (87240, 32)

=== Plage de dates : 2020-01-01 → 2025-12-20
=== Nb villes      : 40
=== Nb régions     : 10
=== Valeurs manquantes totales : 0

=== Types :
code                                   object
id                                      int64
time                           datetime64[ns]
weather_code                            int64
temperature_2m_max                    float64
temperature_2m_min                    float64
temperature_2m_mean                   float64
apparent_temperature_max              float64
apparent_temperature_min              float64
apparent_temperature_mean             float64
daylight_duration                     float64
sunshine_duration                     float64
precipitation_sum                     float64
rain_sum                              float64
snowfall_sum                            int64
precipitation_hours                     int64
wind_speed_10m_max                    float64
wind_gusts_10m_max                    float64
wind_d

,pm2_5,pm10,no2,o3,co,so2,aqi_global
count,87240.0000,87240.0000,87240.0000,87240.0000,87240.0000,87240.0000,87240.0000
mean,29.6700,45.4500,5.1600,62.8600,399.6200,2.3500,85.3800
std,30.9500,47.1600,5.6900,24.9100,340.1000,5.4000,45.9900
min,0.5000,0.7200,0.0000,8.8700,59.6200,0.0000,11.0000
25%,12.0800,17.9600,1.4100,45.1800,248.9100,0.4000,57.0000
50%,20.1400,31.1700,3.6700,57.7400,320.2500,0.9000,72.0000
75%,35.2000,54.7800,6.9000,76.8700,446.4300,1.9700,100.0000
max,972.3500,1344.9500,176.0300,283.8800,17178.6200,57.2500,500.0000


In [6]:
# ── 0.6  Trier correctement (crucial pour les lags) ──────────────────────────
df = df.sort_values(['city_indabax', 'time']).reset_index(drop=True)

# Dictionnaire pour le rapport final
feature_summary = {}

print('Tri par (city_indabax, time) — OK')

Tri par (city_indabax, time) — OK


---
## Section 1 — Features temporelles

In [7]:
# ── 1.1  Features calendaires de base ───────────────────────────────────────
log.info('Section 1 : Features temporelles…')

df['year']         = df['time'].dt.year
df['month']        = df['time'].dt.month
df['day_of_year']  = df['time'].dt.day_of_year          # 1–366
df['week_of_year'] = df['time'].dt.isocalendar().week.astype(int)  # 1–53
df['day_of_month'] = df['time'].dt.day
df['quarter']      = df['time'].dt.quarter

cal_features = ['year', 'month', 'day_of_year', 'week_of_year', 'day_of_month', 'quarter']
feature_summary.update({f: {'category': 'temporal_calendar', 'description': 'Feature calendaire de base'} for f in cal_features})
log.info('1.1 Calendaire : %d features créées', len(cal_features))

20:19:26 [INFO] Section 1 : Features temporelles…
20:19:26 [INFO] 1.1 Calendaire : 6 features créées


In [8]:
# ── 1.2  Encodage cyclique ───────────────────────────────────────────────────
# Sine/cosine pour éviter la discontinuité décembre→janvier, jour365→jour1

df['month_sin']        = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']        = np.cos(2 * np.pi * df['month'] / 12)
df['day_of_year_sin']  = np.sin(2 * np.pi * df['day_of_year'] / 366)
df['day_of_year_cos']  = np.cos(2 * np.pi * df['day_of_year'] / 366)
df['week_sin']         = np.sin(2 * np.pi * df['week_of_year'] / 53)
df['week_cos']         = np.cos(2 * np.pi *
                                 df['week_of_year'] / 53)

cyclic_features = ['month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos', 'week_sin', 'week_cos']
feature_summary.update({f: {'category': 'temporal_cyclic', 'description': 'Encodage cyclique sin/cos'} for f in cyclic_features})
log.info('1.2 Encodage cyclique : %d features créées', len(cyclic_features))

20:19:26 [INFO] 1.2 Encodage cyclique : 6 features créées


In [9]:
# ── 1.3  Saisons camerounaises ───────────────────────────────────────────────
# Classification climatique sub-saharienne propre au Cameroun

def get_cameroon_season(month: int) -> str:
    """Retourne la saison camerounaise en fonction du mois."""
    if month in (11, 12, 1, 2):
        return 'seche'
    elif month in (3, 10):
        return 'transition'
    else:  # 4–9
        return 'humide'

df['season']       = df['month'].map(get_cameroon_season)
df['is_dry_season'] = (df['season'] == 'seche').astype(int)
df['is_wet_season'] = (df['season'] == 'humide').astype(int)
# Harmattan : vents chargés de poussières du Sahara, nov–fév → pics PM2.5/PM10
df['is_harmattan']  = df['is_dry_season'].copy()

season_features = ['season', 'is_dry_season', 'is_wet_season', 'is_harmattan']
feature_summary.update({
    'season':        {'category': 'temporal_season', 'description': 'Saison camerounaise (seche/transition/humide)'},
    'is_dry_season': {'category': 'temporal_season', 'description': 'Flag saison sèche (nov-fév)'},
    'is_wet_season': {'category': 'temporal_season', 'description': 'Flag saison humide (avr-sep)'},
    'is_harmattan':  {'category': 'temporal_season', 'description': 'Flag harmattan (nov-fév) : corrélé PM2.5/PM10'},
})

print('Distribution des saisons :')
print(df['season'].value_counts())
log.info('1.3 Saisons camerounaises : OK')

20:19:26 [INFO] 1.3 Saisons camerounaises : OK


Distribution des saisons :
season
humide        43920
seche         28440
transition    14880
Name: count, dtype: int64


In [10]:
# ── 1.4  Tendance temporelle linéaire ────────────────────────────────────────
df['days_since_start'] = (df['time'] - df['time'].min()).dt.days

feature_summary['days_since_start'] = {
    'category': 'temporal_trend',
    'description': 'Nb jours depuis le début de la série (dérive temporelle)'
}

log.info('1.4 Tendance linéaire : days_since_start 0→%d', df['days_since_start'].max())
print('\n=== Récapitulatif Section 1 ===')
print(f'Features temporelles créées : {len(cal_features) + len(cyclic_features) + len(season_features) + 1}')

20:19:26 [INFO] 1.4 Tendance linéaire : days_since_start 0→2180



=== Récapitulatif Section 1 ===
Features temporelles créées : 17


---
## Section 2 — Features météorologiques dérivées

In [11]:
# ── 2.1  Amplitude thermique ─────────────────────────────────────────────────
log.info('Section 2 : Features météorologiques dérivées…')

df['temp_range'] = df['temperature_2m_max'] - df['temperature_2m_min']
# Forte amplitude → meilleure dispersion des polluants

feature_summary['temp_range'] = {
    'category': 'meteo_derived',
    'description': 'Amplitude thermique journalière (max-min)'
}
log.info('2.1 Amplitude thermique : OK')

20:19:26 [INFO] Section 2 : Features météorologiques dérivées…
20:19:26 [INFO] 2.1 Amplitude thermique : OK


In [12]:
# ── 2.2  Écart température ressentie vs réelle ───────────────────────────────
# Proxy de l'humidité + vent : écart négatif fort = vent fort ou faible humidité

df['apparent_temp_diff_max']  = df['apparent_temperature_max']  - df['temperature_2m_max']
df['apparent_temp_diff_mean'] = df['apparent_temperature_mean'] - df['temperature_2m_mean']

feature_summary.update({
    'apparent_temp_diff_max':  {'category': 'meteo_derived', 'description': 'Écart T ressentie max vs T réelle max'},
    'apparent_temp_diff_mean': {'category': 'meteo_derived', 'description': 'Écart T ressentie moy vs T réelle moy'},
})
log.info('2.2 Écart T ressentie : OK')

20:19:27 [INFO] 2.2 Écart T ressentie : OK


In [13]:
# ── 2.3  Fraction d'ensoleillement ───────────────────────────────────────────
# sunshine_fraction ∈ [0,1]. Proxy de nébulosité.
# O3 fortement corrélé à l'ensoleillement (photochimie).

df['sunshine_fraction'] = df['sunshine_duration'] / (df['daylight_duration'] + 1e-6)
# Clamp à [0,1] par sécurité (erreurs d'arrondi possibles)
df['sunshine_fraction'] = df['sunshine_fraction'].clip(0, 1)

feature_summary['sunshine_fraction'] = {
    'category': 'meteo_derived',
    'description': 'Fraction ensoleillement réel / durée du jour (proxy nébulosité)'
}
log.info('2.3 Fraction ensoleillement : OK')

20:19:27 [INFO] 2.3 Fraction ensoleillement : OK


In [14]:
# ── 2.4  Intensité de la précipitation ───────────────────────────────────────
# La pluie lessive les particules → PM2.5 et PM10 chutent après les pluies

df['precip_intensity'] = df['precipitation_sum'] / (df['precipitation_hours'] + 0.01)
df['precip_flag']      = (df['precipitation_sum'] > 0).astype(int)

feature_summary.update({
    'precip_intensity': {'category': 'meteo_derived', 'description': 'Intensité précipitation (mm/h effectif)'},
    'precip_flag':      {'category': 'meteo_derived', 'description': 'Flag : jour de pluie (0/1)'},
})
log.info('2.4 Précipitations : OK')

20:19:27 [INFO] 2.4 Précipitations : OK


In [15]:
# ── 2.5  Décomposition vectorielle du vent ───────────────────────────────────
# Évite la discontinuité à 0°/360° pour les modèles ML

wind_dir_rad = np.deg2rad(df['wind_direction_10m_dominant'])
df['wind_u'] = df['wind_speed_10m_max'] * np.sin(wind_dir_rad)  # composante E-W
df['wind_v'] = df['wind_speed_10m_max'] * np.cos(wind_dir_rad)  # composante N-S

# Vent calme → mauvaise dispersion → accumulation des polluants
df['calm_wind_flag'] = (df['wind_speed_10m_max'] < 2.0).astype(int)

feature_summary.update({
    'wind_u':        {'category': 'meteo_derived', 'description': 'Composante vent E-O (vectorielle)'},
    'wind_v':        {'category': 'meteo_derived', 'description': 'Composante vent N-S (vectorielle)'},
    'calm_wind_flag':{'category': 'meteo_derived', 'description': 'Flag vent calme (<2 m/s) : accumulation polluants'},
})
log.info('2.5 Vent vectoriel + flag calme : OK')

20:19:27 [INFO] 2.5 Vent vectoriel + flag calme : OK


In [16]:
# ── 2.6  Anomalie ET0 (proxy humidité relative) ───────────────────────────────
# Valeur positive → journée plus sèche que la moyenne du mois pour cette ville

et0_city_month_mean = (
    df.groupby(['city_indabax', 'month'])['et0_fao_evapotranspiration']
      .transform('mean')
)
df['et0_anomaly'] = df['et0_fao_evapotranspiration'] - et0_city_month_mean

feature_summary['et0_anomaly'] = {
    'category': 'meteo_derived',
    'description': 'Anomalie ET0 vs moyenne mois×ville (proxy sécheresse)'
}
log.info('2.6 Anomalie ET0 : OK')

20:19:27 [INFO] 2.6 Anomalie ET0 : OK


In [17]:
# ── 2.7  Rayonnement relatif ─────────────────────────────────────────────────
# MJ/m² par heure effective de jour → corrélé à la formation photochimique de O3

daylight_h = df['daylight_duration'] / 3600  # secondes → heures
df['radiation_per_daylight'] = df['shortwave_radiation_sum'] / (daylight_h + 1e-6)

feature_summary['radiation_per_daylight'] = {
    'category': 'meteo_derived',
    'description': 'Rayonnement court. relatif par heure de jour (MJ/m²/h)'
}

meteo_derived = [
    'temp_range', 'apparent_temp_diff_max', 'apparent_temp_diff_mean',
    'sunshine_fraction', 'precip_intensity', 'precip_flag',
    'wind_u', 'wind_v', 'calm_wind_flag', 'et0_anomaly', 'radiation_per_daylight'
]
log.info('Section 2 terminée : %d features météo dérivées', len(meteo_derived))

20:19:27 [INFO] Section 2 terminée : 11 features météo dérivées


---
## Section 3 — Features de lag (décalées) — CRITIQUE ANTI-LEAKAGE

> **Règle absolue** : `shift(n)` avec `n ≥ 1`, groupé par `city_indabax`, trié par `time`.  
> Jamais de `shift(0)` pour une feature de prédiction.

In [18]:
# ── 3.1  Lags des polluants ───────────────────────────────────────────────────
log.info('Section 3 : Features de lag…')

POLLUTANTS = ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2']
LAG_STEPS  = [1, 2, 3, 7, 14, 30]

lag_cols_created = []

for p in POLLUTANTS:
    for lag in LAG_STEPS:
        col = f'{p}_lag{lag}'
        # groupby city pour ne pas mélanger les séries des différentes villes
        df[col] = df.groupby('city_indabax')[p].transform(lambda s: s.shift(lag))
        lag_cols_created.append(col)
        feature_summary[col] = {
            'category': 'lag_pollutant',
            'description': f'Valeur de {p} à j-{lag} (persistence atmosphérique)'
        }

log.info('3.1 Lags polluants : %d features créées', len(lag_cols_created))

20:19:27 [INFO] Section 3 : Features de lag…
20:19:28 [INFO] 3.1 Lags polluants : 36 features créées


In [19]:
# ── 3.2  Lags de l'AQI global ────────────────────────────────────────────────

for lag in [1, 7]:
    col = f'aqi_global_lag{lag}'
    df[col] = df.groupby('city_indabax')['aqi_global'].transform(lambda s: s.shift(lag))
    lag_cols_created.append(col)
    feature_summary[col] = {
        'category': 'lag_aqi',
        'description': f'AQI global à j-{lag}'
    }

log.info('3.2 Lags AQI global : OK')

20:19:28 [INFO] 3.2 Lags AQI global : OK


In [20]:
# ── 3.3  Lags des variables météo clés ───────────────────────────────────────

meteo_lag_config = {
    'precipitation_sum':      [1, 3],   # effet lessivage décalé
    'wind_speed_10m_max':     [1],
    'temperature_2m_mean':    [1, 3],
}

for col_name, lags in meteo_lag_config.items():
    for lag in lags:
        new_col = f'{col_name}_lag{lag}'
        df[new_col] = df.groupby('city_indabax')[col_name].transform(lambda s, l=lag: s.shift(l))
        lag_cols_created.append(new_col)
        feature_summary[new_col] = {
            'category': 'lag_meteo',
            'description': f'{col_name} à j-{lag}'
        }

log.info('3.3 Lags météo : OK')

20:19:28 [INFO] 3.3 Lags météo : OK


In [21]:
# ── 3.4  Imputation des NaN créés par les lags ────────────────────────────────
# Les lags créent des NaN en début de série (30 premiers jours par ville max).
# On ne supprime pas ces lignes (cibles valides).
# Stratégie : médiane de la ville pour chaque feature de lag.

log.info('NaN avant imputation des lags : %d', df[lag_cols_created].isnull().sum().sum())

for col in lag_cols_created:
    city_medians = df.groupby('city_indabax')[col].transform('median')
    df[col] = df[col].fillna(city_medians)
    # Fallback global si une ville entière est NaN
    df[col] = df[col].fillna(df[col].median())
    feature_summary[col]['nan_after_imputation'] = int(df[col].isnull().sum())

remaining_nan = df[lag_cols_created].isnull().sum().sum()
log.info('NaN après imputation (médiane ville) : %d', remaining_nan)
print(f'Total features de lag créées : {len(lag_cols_created)}')

20:19:28 [INFO] NaN avant imputation des lags : 14360
20:19:29 [INFO] NaN après imputation (médiane ville) : 0


Total features de lag créées : 43


---
## Section 4 — Features de fenêtre glissante (rolling statistics)

> **Anti-leakage** : `shift(1)` avant `rolling()` pour exclure la valeur courante.

In [22]:
# ── Helpers rolling anti-leakage ────────────────────────────────────────────
log.info('Section 4 : Rolling statistics…')

def rolling_stat(series: pd.Series, window: int, stat: str = 'mean') -> pd.Series:
    """
    Calcule une statistique glissante SANS leakage :
    - shift(1) d'abord pour exclure la valeur courante
    - rolling(window) sur le passé uniquement
    min_periods = max(1, window//2) pour les débuts de série
    """
    shifted = series.shift(1)
    min_p   = max(1, window // 2)
    r = shifted.rolling(window=window, min_periods=min_p)
    if stat == 'mean':
        return r.mean()
    elif stat == 'std':
        return r.std()
    elif stat == 'max':
        return r.max()
    elif stat == 'sum':
        return r.sum()
    raise ValueError(f'Stat inconnue : {stat}')

print('Helper rolling_stat défini (shift(1) + rolling, anti-leakage)')

20:19:29 [INFO] Section 4 : Rolling statistics…


Helper rolling_stat défini (shift(1) + rolling, anti-leakage)


In [23]:
# ── 4.1  Rolling mean des polluants ──────────────────────────────────────────
rolling_cols_created = []

WINDOWS_MEAN = [7, 14, 30]

for p in POLLUTANTS:
    for w in WINDOWS_MEAN:
        col = f'{p}_roll{w}_mean'
        df[col] = df.groupby('city_indabax')[p].transform(
            lambda s, ww=w: rolling_stat(s, ww, 'mean')
        )
        rolling_cols_created.append(col)
        feature_summary[col] = {
            'category': 'rolling_mean',
            'description': f'Moyenne glissante {p} sur {w}j (anti-leakage)'
        }

log.info('4.1 Rolling mean : %d features', len(rolling_cols_created))

20:19:29 [INFO] 4.1 Rolling mean : 18 features


In [24]:
# ── 4.2  Rolling std ─────────────────────────────────────────────────────────
std_cols = []
for p in POLLUTANTS:
    col = f'{p}_roll7_std'
    df[col] = df.groupby('city_indabax')[p].transform(
        lambda s: rolling_stat(s, 7, 'std')
    )
    std_cols.append(col)
    feature_summary[col] = {
        'category': 'rolling_std',
        'description': f'Écart-type glissant {p} sur 7j (volatilité locale)'
    }
rolling_cols_created.extend(std_cols)
log.info('4.2 Rolling std : %d features', len(std_cols))

20:19:30 [INFO] 4.2 Rolling std : 6 features


In [25]:
# ── 4.3  Rolling max (pics récents) ──────────────────────────────────────────
max_cols = []
for p in POLLUTANTS:
    col = f'{p}_roll7_max'
    df[col] = df.groupby('city_indabax')[p].transform(
        lambda s: rolling_stat(s, 7, 'max')
    )
    max_cols.append(col)
    feature_summary[col] = {
        'category': 'rolling_max',
        'description': f'Max glissant {p} sur 7j (pics récents)'
    }
rolling_cols_created.extend(max_cols)
log.info('4.3 Rolling max : %d features', len(max_cols))

20:19:30 [INFO] 4.3 Rolling max : 6 features


In [26]:
# ── 4.4  Rolling météo ───────────────────────────────────────────────────────
meteo_roll_config = {
    'precipitation_sum':     ('sum',  7),
    'temperature_2m_mean':   ('mean', 7),
    'wind_speed_10m_max':    ('mean', 7),
    'sunshine_fraction':     ('mean', 7),
}

meteo_roll_cols = []
for col_name, (stat, w) in meteo_roll_config.items():
    new_col = f'{col_name}_roll{w}_{stat}'
    df[new_col] = df.groupby('city_indabax')[col_name].transform(
        lambda s, ss=stat, ww=w: rolling_stat(s, ww, ss)
    )
    meteo_roll_cols.append(new_col)
    feature_summary[new_col] = {
        'category': 'rolling_meteo',
        'description': f'{stat.capitalize()} glissant {col_name} sur {w}j'
    }
rolling_cols_created.extend(meteo_roll_cols)
log.info('4.4 Rolling météo : %d features', len(meteo_roll_cols))

20:19:30 [INFO] 4.4 Rolling météo : 4 features


In [27]:
# ── 4.5  Anomalie par rapport à la moyenne glissante ─────────────────────────
# {p}_anomaly7 = lag1 - roll7_mean : mesure si hier était anormalement pollué

anomaly_cols = []
for p in POLLUTANTS:
    col = f'{p}_anomaly7'
    df[col] = df[f'{p}_lag1'] - df[f'{p}_roll7_mean']
    anomaly_cols.append(col)
    feature_summary[col] = {
        'category': 'rolling_anomaly',
        'description': f'Anomalie {p} : lag1 - moyenne 7j (détrending local)'
    }
rolling_cols_created.extend(anomaly_cols)

# Imputation des NaN issus des rolling (même stratégie : médiane ville)
nan_before = df[rolling_cols_created].isnull().sum().sum()
for col in rolling_cols_created:
    city_medians = df.groupby('city_indabax')[col].transform('median')
    df[col] = df[col].fillna(city_medians)
    df[col] = df[col].fillna(df[col].median())
    feature_summary[col]['nan_after_imputation'] = int(df[col].isnull().sum())

log.info('4.5 Anomalies + imputation rolling : NaN avant=%d, après=%d',
         nan_before, df[rolling_cols_created].isnull().sum().sum())
print(f'Total features rolling créées : {len(rolling_cols_created)}')

20:19:31 [INFO] 4.5 Anomalies + imputation rolling : NaN avant=8640, après=0


Total features rolling créées : 40


---
## Section 5 — Features géographiques et climatiques

In [28]:
# ── 5.1  Encodage cyclique des coordonnées géographiques ────────────────────
log.info('Section 5 : Features géographiques…')

lat_rad = np.deg2rad(df['latitude'])
lon_rad = np.deg2rad(df['longitude'])

df['lat_sin'] = np.sin(lat_rad)
df['lat_cos'] = np.cos(lat_rad)
df['lon_sin'] = np.sin(lon_rad)
df['lon_cos'] = np.cos(lon_rad)

geo_features = ['lat_sin', 'lat_cos', 'lon_sin', 'lon_cos']
feature_summary.update({f: {'category': 'geo_static', 'description': 'Encodage cyclique coordonnées GPS'} for f in geo_features})
log.info('5.1 Encodage lat/lon : OK')

20:19:31 [INFO] Section 5 : Features géographiques…
20:19:31 [INFO] 5.1 Encodage lat/lon : OK


In [29]:
# ── 5.2  Zone éco-climatique du Cameroun ─────────────────────────────────────
# Le Cameroun = "l'Afrique en miniature" : toutes les zones climatiques africaines

eco_zone_map = {
    'Extreme-Nord': 'sahel',    # pas d'accent dans le dataset
    'Nord':         'sahel',
    'Adamaoua':     'soudanien',
    'Nord-Ouest':   'soudanien',
    'Centre':       'guineen',
    'Est':          'guineen',
    'Sud':          'guineen',
    'Ouest':        'guineen',
    'Littoral':     'cotier',
    'Sud-Ouest':    'cotier',
}

df['eco_zone'] = df['region_indabax'].map(eco_zone_map)

# Verification : toutes les regions sont mappees ?
unmapped = df['eco_zone'].isnull().sum()
if unmapped > 0:
    log.warning('%d lignes sans eco_zone -> fallback guineen', unmapped)
    df['eco_zone'] = df['eco_zone'].fillna('guineen')

feature_summary['eco_zone'] = {
    'category': 'geo_climatic',
    'description': 'Zone eco-climatique (sahel/soudanien/guineen/cotier)'
}

print('Distribution des zones eco-climatiques :')
print(df.groupby('eco_zone')['city_indabax'].nunique().rename('nb_villes'))
log.info('5.2 Zones eco-climatiques : OK')


20:19:31 [INFO] 5.2 Zones eco-climatiques : OK


Distribution des zones eco-climatiques :
eco_zone
cotier        8
guineen      16
sahel         8
soudanien     8
Name: nb_villes, dtype: int64


In [30]:
# ── 5.3  Encodage des variables catégorielles ────────────────────────────────

# --- Label Encoding ---
le_city   = LabelEncoder()
le_region = LabelEncoder()
le_eco    = LabelEncoder()

df['city_le']    = le_city.fit_transform(df['city_indabax'])
df['region_le']  = le_region.fit_transform(df['region_indabax'])
df['eco_zone_le']= le_eco.fit_transform(df['eco_zone'])

feature_summary.update({
    'city_le':     {'category': 'encoding_label', 'description': 'Label encoding city_indabax'},
    'region_le':   {'category': 'encoding_label', 'description': 'Label encoding region_indabax'},
    'eco_zone_le': {'category': 'encoding_label', 'description': 'Label encoding eco_zone'},
})

# --- Dummies région (10 catégories) ---
region_dummies = pd.get_dummies(df['region_indabax'], prefix='region', drop_first=False, dtype=int)
df = pd.concat([df, region_dummies], axis=1)
for col in region_dummies.columns:
    feature_summary[col] = {'category': 'encoding_dummy', 'description': f'Dummy région : {col}'}

# --- Dummies eco_zone ---
eco_dummies = pd.get_dummies(df['eco_zone'], prefix='eco', drop_first=False, dtype=int)
df = pd.concat([df, eco_dummies], axis=1)
for col in eco_dummies.columns:
    feature_summary[col] = {'category': 'encoding_dummy', 'description': f'Dummy eco_zone : {col}'}

log.info('5.3 Label encoding + dummies : %d régions, %d zones', 
         len(region_dummies.columns), len(eco_dummies.columns))

20:19:31 [INFO] 5.3 Label encoding + dummies : 10 régions, 4 zones


In [31]:
# ── 5.4  Target Encoding avec smoothing ─────────────────────────────────────
# mean encoding + smoothing Bayésien : évite l'overfitting sur les petites villes
# target_enc = (count × mean_city + global_mean × alpha) / (count + alpha)

ALPHA = 10  # paramètre de lissage

te_cols_created = []
for target in TARGETS:
    global_mean = df[target].mean()
    city_stats  = df.groupby('city_indabax')[target].agg(['mean', 'count'])
    city_stats['smooth_enc'] = (
        city_stats['count'] * city_stats['mean'] + global_mean * ALPHA
    ) / (city_stats['count'] + ALPHA)
    col = f'city_te_{target}'
    df[col] = df['city_indabax'].map(city_stats['smooth_enc'])
    te_cols_created.append(col)
    feature_summary[col] = {
        'category': 'target_encoding',
        'description': f'Target encoding (smoothed) city → {target} (alpha={ALPHA})'
    }

log.info('5.4 Target encoding (smoothed, alpha=%d) : %d features', ALPHA, len(te_cols_created))
print(f'Section 5 terminée. Colonnes df : {df.shape[1]}')

20:19:32 [INFO] 5.4 Target encoding (smoothed, alpha=10) : 7 features


Section 5 terminée. Colonnes df : 172


---
## Section 6 — Features d'interaction et non-linéaires

In [32]:
# ── 6.1  Interactions météo × saison ─────────────────────────────────────────
log.info('Section 6 : Features d\'interaction…')

# chaleur sèche → intensifie O3 et poussières
df['temp_x_dry'] = df['temperature_2m_mean'] * df['is_dry_season']

# index de stress photochimique → formation O3
df['rad_x_sunshine'] = df['shortwave_radiation_sum'] * df['sunshine_fraction']

# vent + pluie = bon lessivage des particules
df['wind_x_precip'] = df['wind_speed_10m_max'] * df['precip_flag']

# pire configuration pour PM2.5 : vent calme + saison sèche
df['calm_x_dry'] = df['calm_wind_flag'] * df['is_dry_season']

inter1 = ['temp_x_dry', 'rad_x_sunshine', 'wind_x_precip', 'calm_x_dry']
feature_summary.update({
    'temp_x_dry':      {'category': 'interaction', 'description': 'T_mean × is_dry_season (chaleur sèche → O3/PM)'},
    'rad_x_sunshine':  {'category': 'interaction', 'description': 'Rayonnement × fraction soleil (stress photochimique O3)'},
    'wind_x_precip':   {'category': 'interaction', 'description': 'Vent × pluie (lessivage particules)'},
    'calm_x_dry':      {'category': 'interaction', 'description': 'Vent calme × saison sèche (pire cas PM2.5)'},
})
log.info('6.1 Interactions météo×saison : %d features', len(inter1))

20:19:32 [INFO] Section 6 : Features d'interaction…
20:19:32 [INFO] 6.1 Interactions météo×saison : 4 features


In [33]:
# ── 6.2  Interaction géographie × météo ──────────────────────────────────────

# Gradient thermique N-S
df['lat_x_temp_range'] = df['latitude'] * df['temp_range']

# Zone sahel × cumul pluviométrique 7j (désert + rare pluie → extrêmes)
df['sahel_x_precip7'] = df['eco_sahel'] * df['precipitation_sum_roll7_sum']

inter2 = ['lat_x_temp_range', 'sahel_x_precip7']
feature_summary.update({
    'lat_x_temp_range': {'category': 'interaction', 'description': 'Latitude × amplitude thermique (gradient N-S)'},
    'sahel_x_precip7':  {'category': 'interaction', 'description': 'Sahel × précip 7j (désert + rare pluie)'},
})
log.info('6.2 Interactions géo×météo : %d features', len(inter2))

20:19:32 [INFO] 6.2 Interactions géo×météo : 2 features


In [34]:
# ── 6.3  Features non-linéaires ──────────────────────────────────────────────

# Transformations log1p : relation précipitations-PM est logarithmique
df['log1p_precip'] = np.log1p(df['precipitation_sum'])
df['log1p_wind']   = np.log1p(df['wind_speed_10m_max'])

# Terme quadratique de la température (effets non-linéaires)
df['temp_mean_sq'] = df['temperature_2m_mean'] ** 2

nonlinear = ['log1p_precip', 'log1p_wind', 'temp_mean_sq']
feature_summary.update({
    'log1p_precip': {'category': 'nonlinear', 'description': 'log1p(precipitation_sum) : relation log PM-pluie'},
    'log1p_wind':   {'category': 'nonlinear', 'description': 'log1p(wind_speed_10m_max)'},
    'temp_mean_sq': {'category': 'nonlinear', 'description': 'T_mean² (effets quadratiques)'},
})

log.info('6.3 Non-linéaires : %d features', len(nonlinear))
print(f'Section 6 terminée. Colonnes df : {df.shape[1]}')

20:19:32 [INFO] 6.3 Non-linéaires : 3 features


Section 6 terminée. Colonnes df : 181


---
## Section 7 — Analyse qualité et sélection préliminaire

In [35]:
# ── 7.0  Identifier les colonnes features vs identifiants vs cibles ──────────
log.info('Section 7 : Analyse qualité…')

ID_COLS     = ['code', 'id', 'time', 'city_indabax', 'region_indabax', 
               'latitude', 'longitude', 'season', 'eco_zone']
FEATURE_COLS = [c for c in df.columns 
                if c not in ID_COLS + TARGETS
                and df[c].dtype != object]

print(f'Colonnes identifiants  : {len(ID_COLS)}')
print(f'Colonnes cibles (TARGETS) : {len(TARGETS)}')
print(f'Colonnes features numériques : {len(FEATURE_COLS)}')
print(f'Total colonnes df      : {df.shape[1]}')

20:19:32 [INFO] Section 7 : Analyse qualité…


Colonnes identifiants  : 9
Colonnes cibles (TARGETS) : 7
Colonnes features numériques : 165
Total colonnes df      : 181


In [36]:
# ── 7.1  Corrélations de Pearson avec les cibles ─────────────────────────────
# Calcul de la matrice de corrélation feature → cibles

corr_data = df[FEATURE_COLS + TARGETS].corr(method='pearson')
corr_targets = corr_data.loc[FEATURE_COLS, TARGETS].abs()

# Top 20 features pour pm2_5 et aqi_global
top20_pm25   = corr_targets['pm2_5'].nlargest(20)
top20_aqi    = corr_targets['aqi_global'].nlargest(20)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
top20_pm25.sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 20 features — corrélation |Pearson| avec PM2.5', fontsize=12)
axes[0].set_xlabel('|Corrélation|')

top20_aqi.sort_values().plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Top 20 features — corrélation |Pearson| avec AQI Global', fontsize=12)
axes[1].set_xlabel('|Corrélation|')

plt.tight_layout()
plt.savefig(DATA_OUT/ 'fig_corr_top20_features.png', dpi=110, bbox_inches='tight')
plt.show()
log.info('7.1 Corrélations Pearson calculées et sauvegardées')

20:19:46 [INFO] 7.1 Corrélations Pearson calculées et sauvegardées


In [37]:
# ── 7.1b  Heatmap corrélations top features × toutes cibles ──────────────────
# Union des top 15 features pour pm2_5 et aqi_global

top_feats = list(set(top20_pm25.nlargest(15).index.tolist() + 
                     top20_aqi.nlargest(15).index.tolist()))

fig, ax = plt.subplots(figsize=(12, max(6, len(top_feats) * 0.4)))
sns.heatmap(
    corr_data.loc[top_feats, TARGETS].round(2),
    annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax
)
ax.set_title('Corrélations : top features × 7 cibles', fontsize=13)
plt.tight_layout()
plt.savefig(DATA_OUT / 'fig_heatmap_feature_target_corr.png', dpi=110, bbox_inches='tight')
plt.show()

In [38]:
# ── 7.2  Détection multicolinéarité (VIF) ────────────────────────────────────
# Calculer VIF sur un sous-ensemble pour la performance (max 50 features)
# On prend les 50 features les plus corrélées avec pm2_5

log.info('7.2 Calcul VIF (sous-échantillon 50 features)…')

vif_features = corr_targets['pm2_5'].nlargest(50).index.tolist()
vif_data     = df[vif_features].dropna().sample(min(5000, len(df)), random_state=42)

vif_results = []
for i, col in enumerate(vif_features):
    try:
        vif_val = variance_inflation_factor(vif_data.values, i)
        vif_results.append({'feature': col, 'VIF': round(vif_val, 2)})
    except Exception:
        vif_results.append({'feature': col, 'VIF': np.nan})

vif_df = pd.DataFrame(vif_results).sort_values('VIF', ascending=False)
high_vif = vif_df[vif_df['VIF'] > 10]

print(f'Features avec VIF > 10 : {len(high_vif)}')
print(high_vif.head(20).to_string(index=False))

# Documenter dans feature_summary (pas de suppression, tree-based OK)
for _, row in high_vif.iterrows():
    if row['feature'] in feature_summary:
        feature_summary[row['feature']]['vif'] = row['VIF']
        feature_summary[row['feature']]['high_multicolinearity'] = True

log.info('VIF calculé. %d features avec VIF>10 (documentées, non supprimées)', len(high_vif))

20:19:47 [INFO] 7.2 Calcul VIF (sous-échantillon 50 features)…
20:19:49 [INFO] VIF calculé. 48 features avec VIF>10 (documentées, non supprimées)


Features avec VIF > 10 : 48
          feature       VIF
   pm10_roll7_max 1360.6200
  pm2_5_roll7_max 1321.1900
  pm10_roll7_mean 1161.1700
 pm2_5_roll7_mean 1142.8800
 pm10_roll14_mean  852.0400
pm2_5_roll14_mean  765.0300
 pm10_roll30_mean  531.2600
pm2_5_roll30_mean  451.0800
         week_cos  434.0400
  day_of_year_cos  432.4200
    co_roll7_mean  389.7900
     co_roll7_max  378.1500
   co_roll14_mean  346.1600
   pm10_roll7_std  267.1000
  pm2_5_roll7_std  263.8900
  no2_roll14_mean  251.3000
       pm2_5_lag3  225.4700
        pm10_lag3  214.3400
   no2_roll7_mean  207.3400
    no2_roll7_max  200.8800


In [39]:
# ── 7.3  Vérification anti-leakage finale ────────────────────────────────────
log.info('7.3 Vérification anti-leakage finale…')

issues = []

# 1) Les cibles ne doivent pas apparaître comme features (sauf via lag/rolling)
for t in TARGETS:
    if t in FEATURE_COLS:
        issues.append(f'LEAKAGE DIRECT : {t} dans FEATURE_COLS')

# 2) Vérifier que les colonnes lag ne sont pas shift(0)
#    Proxy : vérifier que les lag_cols ne sont pas identiques aux originaux
for p in POLLUTANTS:
    lag1_col = f'{p}_lag1'
    corr_val = df[[p, lag1_col]].corr().iloc[0, 1]
    if corr_val > 0.9999:
        issues.append(f'LEAKAGE POTENTIEL : {lag1_col} corrélation parfaite avec {p}')

if issues:
    log.error('PROBLÈMES DÉTECTÉS :\n' + '\n'.join(issues))
    for i in issues:
        print(f'[ERREUR] {i}')
else:
    log.info('Anti-leakage check PASSED : aucun problème détecté')
    print('[OK] Vérification anti-leakage : aucun problème détecté')

# Bilan des features par catégorie
cat_counts = {}
for fname, finfo in feature_summary.items():
    cat = finfo.get('category', 'unknown')
    cat_counts[cat] = cat_counts.get(cat, 0) + 1

print('\n=== Features par catégorie ===')
for cat, cnt in sorted(cat_counts.items()):
    print(f'  {cat:<30} : {cnt}')
print(f'\n  TOTAL : {sum(cat_counts.values())}')

20:19:49 [INFO] 7.3 Vérification anti-leakage finale…
20:19:49 [INFO] Anti-leakage check PASSED : aucun problème détecté


[OK] Vérification anti-leakage : aucun problème détecté

=== Features par catégorie ===
  encoding_dummy                 : 14
  encoding_label                 : 3
  geo_climatic                   : 1
  geo_static                     : 4
  interaction                    : 6
  lag_aqi                        : 2
  lag_meteo                      : 5
  lag_pollutant                  : 36
  meteo_derived                  : 11
  nonlinear                      : 3
  rolling_anomaly                : 6
  rolling_max                    : 6
  rolling_mean                   : 18
  rolling_meteo                  : 4
  rolling_std                    : 6
  target_encoding                : 7
  temporal_calendar              : 6
  temporal_cyclic                : 6
  temporal_season                : 4
  temporal_trend                 : 1

  TOTAL : 149


---
## Section 8 — Sauvegarde et rapport de synthèse

In [40]:
# ── FE 8.1  Feature Engineering — résumé en mémoire (pas de sauvegarde disque) ──
log.info('Section 8 : Feature Engineering terminé — df en mémoire')

# Pas de sauvegarde parquet/xlsx ici : le df est directement exploité en mémoire
# par les sections suivantes. Cela évite les I/O lourds inutiles.

log.info('Shape final FE : %s | NaN total : %d', df.shape, df.isnull().sum().sum())
print(f'Shape : {df.shape}')
print(f'Mémoire estimée : {df.memory_usage(deep=True).sum() / 1e6:.1f} Mo (en RAM)')


20:19:49 [INFO] Section 8 : Feature Engineering terminé — df en mémoire
20:19:49 [INFO] Shape final FE : (87240, 181) | NaN total : 0


Shape : (87240, 181)
Mémoire estimée : 145.8 Mo (en RAM)


In [41]:
# ── 8.2  Rapport de synthèse ─────────────────────────────────────────────────

n_original = 32  # colonnes du dataset brut
n_features_created = len(feature_summary)

print('=' * 65)
print('  RAPPORT DE SYNTHÈSE — FEATURE ENGINEERING')
print('  Hackathon IndabaX Cameroon 2026')
print('=' * 65)

print(f'\n[DATASET]')
print(f'  Période          : {df["time"].min().date()} → {df["time"].max().date()}')
print(f'  Nb observations  : {len(df):,}')
print(f'  Nb villes        : {df["city_indabax"].nunique()}')
print(f'  Nb régions       : {df["region_indabax"].nunique()}')

print(f'\n[FEATURES]')
print(f'  Colonnes brutes  : {n_original}')
print(f'  Features créées  : {n_features_created}')
print(f'  Total colonnes   : {df.shape[1]}')
print(f'\n  Détail par catégorie :')
for cat, cnt in sorted(cat_counts.items()):
    print(f'    {cat:<32} : {cnt:3d}')

print(f'\n[TOP 5 FEATURES — corrélation |Pearson| avec PM2.5]')
for feat, val in top20_pm25.nlargest(5).items():
    print(f'    {feat:<35} : {val:.4f}')

print(f'\n[TOP 5 FEATURES — corrélation |Pearson| avec AQI Global]')
for feat, val in top20_aqi.nlargest(5).items():
    print(f'    {feat:<35} : {val:.4f}')

print(f'\n[VALEURS MANQUANTES]')
nan_total_final = df[FEATURE_COLS].isnull().sum().sum()
print(f'  NaN total dans les features (après imputation) : {nan_total_final}')
print(f'  NaN total dans les cibles                      : {df[TARGETS].isnull().sum().sum()}')


print('\n' + '=' * 65)
print('  Feature Engineering terminé avec succès.')
print('  Le df enrichi est directement disponible en mémoire pour la modélisation.')
print('=' * 65)

  RAPPORT DE SYNTHÈSE — FEATURE ENGINEERING
  Hackathon IndabaX Cameroon 2026

[DATASET]
  Période          : 2020-01-01 → 2025-12-20
  Nb observations  : 87,240
  Nb villes        : 40
  Nb régions       : 10

[FEATURES]
  Colonnes brutes  : 32
  Features créées  : 149
  Total colonnes   : 181

  Détail par catégorie :
    encoding_dummy                   :  14
    encoding_label                   :   3
    geo_climatic                     :   1
    geo_static                       :   4
    interaction                      :   6
    lag_aqi                          :   2
    lag_meteo                        :   5
    lag_pollutant                    :  36
    meteo_derived                    :  11
    nonlinear                        :   3
    rolling_anomaly                  :   6
    rolling_max                      :   6
    rolling_mean                     :  18
    rolling_meteo                    :   4
    rolling_std                      :   6
    target_encoding              

---
## Section 9 — Features avancées (amélioration des modèles)

Features supplémentaires à fort impact physique : proxy de ventilation,
score de risque pollution, séquences sèches, transport régional, rangs percentiles.

In [42]:
# ── FE 9.0  Vérification état du df (exécution séquentielle en mémoire) ────────
# Pas de checkpoint disque : si df est déjà en mémoire avec les features avancées,
# on skipe le recalcul. Sinon on continue avec le df de la section 8.

if 'ventilation_proxy' not in df.columns:
    log.info('Section 9 : calcul des features avancées sur df existant')
else:
    log.info('Section 9 : features avancées déjà présentes dans df — OK')

POLLUTANTS = ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2']
if 'feature_summary' not in dir():
    feature_summary = {}

log.info('Shape: %s | NaN: %d', df.shape, df.isnull().sum().sum())
print(f'Shape: {df.shape}')


20:19:50 [INFO] Section 9 : calcul des features avancées sur df existant
20:19:50 [INFO] Shape: (87240, 181) | NaN: 0


Shape: (87240, 181)


In [43]:
# ── 9.1  Proxy de ventilation (dispersion des polluants) ─────────────────────
# facteur de ventilation = vitesse du vent × ET0 (hauteur couche limite proxy)
# Valeur élevée -> bonne dispersion -> moins de pollution

if 'ventilation_proxy' not in df.columns:
    df['ventilation_proxy'] = df['wind_speed_10m_max'] * df['et0_fao_evapotranspiration']
    feature_summary['ventilation_proxy'] = {
        'category': 'advanced_interaction',
        'description': 'vent × ET0 : proxy dispersion atmospherique des polluants'
    }
    log.info('9.1 ventilation_proxy : OK')
else:
    log.info('[SKIP] ventilation_proxy existe')

20:19:50 [INFO] 9.1 ventilation_proxy : OK


In [44]:
# ── 9.2  Score de risque pollution PM2.5 ─────────────────────────────────────
# Pire configuration : polluant déjà élevé hier + vent calme + harmattan
# Ce score capture les épisodes de pollution intense

if 'pollution_risk_score' not in df.columns:
    df['pollution_risk_score'] = (
        df['pm2_5_lag1'] *
        df['calm_wind_flag'] *
        df['is_harmattan']
    )
    feature_summary['pollution_risk_score'] = {
        'category': 'advanced_interaction',
        'description': 'pm2_5_lag1 x calm_wind x is_harmattan : score risque pollution'
    }
    log.info('9.2 pollution_risk_score : OK')
else:
    log.info('[SKIP] pollution_risk_score existe')

20:19:50 [INFO] 9.2 pollution_risk_score : OK


In [45]:
# ── 9.3  Séquences de jours secs consécutifs ─────────────────────────────────
# Plus une ville reste sans pluie, plus les polluants s'accumulent.
# Cette feature capture la dynamique d'accumulation progressive.

if 'dry_days_streak' not in df.columns:
    def compute_dry_streak(precip_flag_series):
        """Nombre de jours secs consécutifs AVANT le jour courant (shift anti-leakage)."""
        # shift(1) : on ne compte pas le jour courant
        shifted = precip_flag_series.shift(1).fillna(0)
        # Grouper les séquences
        streak = []
        count  = 0
        for v in shifted:
            if v == 0:   # jour sec (pas de pluie)
                count += 1
            else:
                count = 0
            streak.append(count)
        return pd.Series(streak, index=precip_flag_series.index)

    df['dry_days_streak'] = df.groupby('city_indabax')['precip_flag'].transform(
        compute_dry_streak
    )
    feature_summary['dry_days_streak'] = {
        'category': 'advanced_temporal',
        'description': 'Nb jours secs consecutifs avant le jour J (accumulation polluants)'
    }
    log.info('9.3 dry_days_streak : max=%d', df['dry_days_streak'].max())
else:
    log.info('[SKIP] dry_days_streak existe')

20:19:50 [INFO] 9.3 dry_days_streak : max=210


In [46]:
# ── 9.4  Transport régional : pollution moyenne régionale décalée ─────────────
# Proxy du transport inter-villes : si les villes voisines sont polluées,
# la pollution peut se propager (vent dominant).

if 'region_pm25_lag1' not in df.columns:
    # Moyenne régionale AVANT calcul du lag -> pas de leakage
    region_daily_mean = (
        df.groupby(['region_indabax', 'time'])['pm2_5']
        .transform('mean')
    )
    df['region_pm25_mean'] = region_daily_mean
    df['region_pm25_lag1'] = df.groupby('city_indabax')['region_pm25_mean']         .transform(lambda s: s.shift(1))

    # Pareil pour pm10 (poussières)
    region_pm10_mean = df.groupby(['region_indabax', 'time'])['pm10'].transform('mean')
    df['region_pm10_mean'] = region_pm10_mean
    df['region_pm10_lag1'] = df.groupby('city_indabax')['region_pm10_mean']         .transform(lambda s: s.shift(1))

    # Imputation NaN (début de série)
    for col in ['region_pm25_lag1', 'region_pm10_lag1']:
        med = df.groupby('city_indabax')[col].transform('median')
        df[col] = df[col].fillna(med).fillna(df[col].median())
        feature_summary[col] = {
            'category': 'advanced_spatial',
            'description': f'Moyenne regionale {col.split("_")[1]} a j-1 (transport inter-villes)'
        }

    # Nettoyer colonnes intermédiaires
    df.drop(columns=['region_pm25_mean', 'region_pm10_mean'], inplace=True)
    log.info('9.4 Transport regional (region_pm25_lag1, region_pm10_lag1) : OK')
else:
    log.info('[SKIP] region_pm25_lag1 existe')

20:19:50 [INFO] 9.4 Transport regional (region_pm25_lag1, region_pm10_lag1) : OK


In [47]:
# ── 9.5  Rangs percentiles (robustesse aux outliers) ─────────────────────────
# Rang de la valeur du polluant dans la distribution historique de la ville.
# Utiliser uniquement le passé (expanding window par groupe).
# La valeur du rang est moins sensible aux valeurs extrêmes.

pct_cols = ['pm2_5', 'pm10', 'co', 'wind_speed_10m_max', 'precipitation_sum']

for col in pct_cols:
    new_col = f'{col}_pct_city'
    if new_col not in df.columns:
        # Expanding rank sur le train pour éviter le leakage global
        df[new_col] = df.groupby('city_indabax')[col].transform(
            lambda s: s.expanding().rank(pct=True)
        )
        feature_summary[new_col] = {
            'category': 'advanced_rank',
            'description': f'Rang percentile de {col} dans l historique de la ville'
        }

log.info('9.5 Percentile ranks : %d features', len(pct_cols))

20:19:51 [INFO] 9.5 Percentile ranks : 5 features


In [48]:
# ── 9.6  Interaction saison × zone + intensité sèche ────────────────────────
if 'harmattan_intensity' not in df.columns:
    # Intensité harmattan = cumul jours secs × score risque (normalisé)
    df['harmattan_intensity'] = df['dry_days_streak'] * df['is_harmattan']

    # Interaction chaleur × ventilation (pire pour O3 quand faible ventilation)
    df['heat_no_wind']  = df['temperature_2m_mean'] * (1 - df['calm_wind_flag'])
    df['o3_risk_score'] = df['sunshine_fraction'] * df['temperature_2m_mean'] * df['is_dry_season']

    new_adv = ['harmattan_intensity', 'heat_no_wind', 'o3_risk_score']
    for c in new_adv:
        feature_summary[c] = {
            'category': 'advanced_interaction',
            'description': f'Feature interaction avancee : {c}'
        }
    log.info('9.6 Interactions avancees : %d features', len(new_adv))
else:
    log.info('[SKIP] harmattan_intensity existe')

20:19:51 [INFO] 9.6 Interactions avancees : 3 features


In [49]:
# ── 9.7  Vérification anti-leakage + sauvegarde ──────────────────────────────
# Verifier que les nouvelles features ne contiennent pas les cibles directes
new_feat_cols = [
    'ventilation_proxy', 'pollution_risk_score', 'dry_days_streak',
    'region_pm25_lag1', 'region_pm10_lag1',
    'harmattan_intensity', 'heat_no_wind', 'o3_risk_score',
] + [f'{c}_pct_city' for c in ['pm2_5','pm10','co','wind_speed_10m_max','precipitation_sum']]

new_feat_cols = [c for c in new_feat_cols if c in df.columns]

nan_count = df[new_feat_cols].isnull().sum().sum()
log.info('NaN dans nouvelles features : %d', nan_count)

# Imputation finale
for col in new_feat_cols:
    if df[col].isnull().sum() > 0:
        med = df.groupby('city_indabax')[col].transform('median')
        df[col] = df[col].fillna(med).fillna(df[col].median())

print(f'Nouvelles features ajoutees : {len(new_feat_cols)}')
print(f'Shape final : {df.shape}')
print(f'NaN total   : {df.isnull().sum().sum()}')

20:19:51 [INFO] NaN dans nouvelles features : 0


Nouvelles features ajoutees : 13
Shape final : (87240, 194)
NaN total   : 0


In [50]:
# ── FE 9.8  Résumé Section 9 (pas de sauvegarde disque) ─────────────────────
log.info('Section 9 terminée — df conservé en mémoire')

new_feat_cols_in_df = [c for c in df.columns if c in new_feat_cols] if 'new_feat_cols' in dir() else []
print(f'Dataset final : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'Nouvelles features (Section 9) : {len(new_feat_cols_in_df)}')
print(f'Mémoire estimée : {df.memory_usage(deep=True).sum() / 1e6:.1f} Mo (en RAM)')


20:19:51 [INFO] Section 9 terminée — df conservé en mémoire


Dataset final : 87,240 lignes × 194 colonnes
Nouvelles features (Section 9) : 13
Mémoire estimée : 154.9 Mo (en RAM)


## Section 1 — Préparation des données

In [51]:
# ── 1.0  df déjà en mémoire (produit par la section FE ci-dessus) ─────────────
# Le chargement depuis parquet est omis : on utilise directement le df enrichi.
log.info('df en mémoire : %s | période %s → %s | %d villes',
         df.shape,
         df.time.min().date(), df.time.max().date(),
         df.city_indabax.nunique())
df[TARGETS].describe().round(3)

20:19:51 [INFO] df en mémoire : (87240, 194) | période 2020-01-01 → 2025-12-20 | 40 villes


,pm2_5,pm10,no2,o3,co,so2,aqi_global
count,87240.0000,87240.0000,87240.0000,87240.0000,87240.0000,87240.0000,87240.0000
mean,29.6730,45.4480,5.1600,62.8610,399.6150,2.3540,85.3830
std,30.9540,47.1630,5.6850,24.9060,340.1040,5.3950,45.9930
min,0.5000,0.7210,0.0000,8.8750,59.6250,0.0000,11.0000
25%,12.0830,17.9620,1.4120,45.1790,248.9060,0.4000,57.0000
50%,20.1380,31.1670,3.6690,57.7380,320.2500,0.9000,72.0000
75%,35.1970,54.7770,6.9000,76.8730,446.4300,1.9730,100.0000
max,972.3510,1344.9510,176.0320,283.8750,17178.6220,57.2520,500.0000


In [52]:
# ── 1.1 Sélection des features ─────────────────────────────────────────────────
EXCLUDE = (['code', 'id', 'time', 'city_indabax', 'region_indabax',
            'latitude', 'longitude', 'season', 'eco_zone'] + TARGETS)
FEATURE_COLS = [c for c in df.columns
                if c not in EXCLUDE and df[c].dtype != object]
log.info('Features retenues : %d', len(FEATURE_COLS))

# ── 1.2 Split temporel ─────────────────────────────────────────────────────────
df_train    = df[df['time'] <= TRAIN_END].copy().reset_index(drop=True)
df_valid    = df[(df['time'] >= VALID_START) & (df['time'] <= VALID_END)].copy().reset_index(drop=True)
df_test     = df[df['time'] >= TEST_START].copy().reset_index(drop=True)
df_trainval = pd.concat([df_train, df_valid], ignore_index=True)

# Vérification absence de chevauchement
assert df_train['time'].max() < pd.Timestamp(VALID_START)
assert df_valid['time'].max() < pd.Timestamp(TEST_START)
assert df_valid['time'].min() > df_train['time'].max()

X_train    = df_train[FEATURE_COLS].values.astype(np.float32)
X_valid    = df_valid[FEATURE_COLS].values.astype(np.float32)
X_test     = df_test[FEATURE_COLS].values.astype(np.float32)
X_trainval = df_trainval[FEATURE_COLS].values.astype(np.float32)

log.info('Split  train=%d  valid=%d  test=%d  trainval=%d',
         len(df_train), len(df_valid), len(df_test), len(df_trainval))

# ── 1.3 Log-transform des cibles asymétriques ─────────────────────────────────
y_true_train = {t: df_train[t].values for t in TARGETS}
y_true_valid = {t: df_valid[t].values for t in TARGETS}
y_true_test  = {t: df_test[t].values  for t in TARGETS}

def log_transform(y, target):
    return np.log1p(np.clip(y, 0, None)) if target in TARGETS_LOG else y

def inv_transform(y, target):
    return np.clip(np.expm1(np.clip(y, -10, 20)), 0, None) if target in TARGETS_LOG else np.clip(y, 0, None)

y_train_log = {t: log_transform(y_true_train[t], t) for t in TARGETS}
y_valid_log = {t: log_transform(y_true_valid[t], t) for t in TARGETS}
y_test_log  = {t: log_transform(y_true_test[t],  t) for t in TARGETS}
y_trainval_log = {t: log_transform(
    np.concatenate([y_true_train[t], y_true_valid[t]]), t) for t in TARGETS}

# ── 1.4 TimeSeriesSplit ────────────────────────────────────────────────────────
tscv = TimeSeriesSplit(n_splits=N_FOLDS, gap=30)
log.info('TimeSeriesSplit : %d folds, gap=30 jours', N_FOLDS)


20:19:52 [INFO] Features retenues : 178
20:19:52 [INFO] Split  train=69400  valid=3680  test=14160  trainval=73080
20:19:52 [INFO] TimeSeriesSplit : 5 folds, gap=30 jours


In [53]:
# ── 1.5 Fonction evaluate() commune ───────────────────────────────────────────
def evaluate(y_true, y_pred, target_name):
    """
    Calcule les métriques sur les valeurs en ECHELLE ORIGINALE.
    y_true / y_pred doivent être déjà inverse-transformés.
    """
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    mbe  = float(np.mean(y_pred - y_true))
    thresh = MAPE_THRESHOLDS.get(target_name, 1.0)
    mask   = y_true > thresh
    mape   = (float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
              if mask.sum() >= 0.1 * len(y_true) else float('nan'))
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape, 'MBE': mbe}


def print_table(title, data_dict, metric='RMSE', algos=None, show_obj=True):
    """Affiche un tableau comparatif metric par algo et par cible."""
    if algos is None:
        algos = list(data_dict.keys())
    algos = [a for a in algos if a in data_dict]
    w = 10
    header = f'  {"Cible":<13}' + ''.join(f'  {a[:w]:>{w}}' for a in algos)
    if show_obj:
        header += f'  {"Objectif":>10}  {"NB06":>8}'
    print('=' * (len(header) + 2))
    print(f'  {title}  —  {metric}')
    print('=' * (len(header) + 2))
    print(header)
    print('  ' + '-' * (len(header) - 2))
    for t in TARGETS:
        row = f'  {t:<13}'
        best_v, best_a = float('inf'), None
        for a in algos:
            v = data_dict[a].get(t, {}).get(metric, float('nan'))
            if not np.isnan(v) and v < best_v:
                best_v, best_a = v, a
        for a in algos:
            v = data_dict[a].get(t, {}).get(metric, float('nan'))
            s = f'{v:>{w}.4f}' if not np.isnan(v) else f'{"N/A":>{w}}'
            row += f'  {s}'
            if a == best_a:
                row = row[:-len(s)-2] + f'  ★{s[1:]:>{w-1}}'
        if show_obj:
            row += f'  {OBJ_RMSE[t]:>10.2f}  {NB06_RMSE[t]:>8.2f}'
        print(row)
    print('=' * (len(header) + 2))

log.info('Section 1 terminee — evaluate() et print_table() pretes')


20:19:52 [INFO] Section 1 terminee — evaluate() et print_table() pretes


## Section 2 — Baselines

In [54]:
log.info('=== Section 2 : Baselines ===')
results_dict['bl_persistence'] = {}
results_dict['bl_city_month']  = {}
all_test_preds['bl_persistence'] = {}
all_test_preds['bl_city_month']  = {}

for t in TARGETS:
    # BL1 — Persistence (lag-1)
    lc = 'aqi_global_lag1' if t == 'aqi_global' else f'{t}_lag1'
    if lc in df_test.columns:
        yp_pers = df_test[lc].values.clip(0, None)
    else:
        yp_pers = np.full(len(df_test), y_true_train[t].mean())

    # BL2 — Moyenne (city × month) sur le train
    cm_mean = (df_train.groupby(['city_indabax', 'month'])[t].mean()
               .rename('cm_mean').reset_index())
    yp_cm = (df_test
             .merge(cm_mean, on=['city_indabax', 'month'], how='left')['cm_mean']
             .fillna(y_true_train[t].mean()).values)

    results_dict['bl_persistence'][t] = evaluate(y_true_test[t], yp_pers, t)
    results_dict['bl_city_month'][t]   = evaluate(y_true_test[t], yp_cm,   t)
    all_test_preds['bl_persistence'][t] = yp_pers
    all_test_preds['bl_city_month'][t]  = yp_cm

print_table('BASELINES — RMSE test 2025', results_dict,
            algos=['bl_persistence', 'bl_city_month'], show_obj=False)


20:19:52 [INFO] === Section 2 : Baselines ===


  BASELINES — RMSE test 2025  —  RMSE
  Cible          bl_persist  bl_city_mo
  -------------------------------------
  pm2_5          ★   6.0116     17.1986
  pm10           ★  16.1230     28.4067
  no2            ★   2.8462      5.2151
  o3             ★   9.0690     24.8107
  co             ★  88.7512    205.0482
  so2            ★   0.5660      4.3490
  aqi_global     ★  13.4829     26.0452


## Section 3 — LightGBM (référence NB06)

In [55]:
log.info('=== Section 3 : LightGBM ===')

LGBM_COMMON = dict(
    n_estimators=3000, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0, n_jobs=-1, verbose=-1,
    random_state=RANDOM_STATE,
)
LGBM_PER = {
    'pm2_5'     : dict(num_leaves=127, learning_rate=0.04, min_child_samples=30, max_depth=8),
    'pm10'      : dict(num_leaves=127, learning_rate=0.04, min_child_samples=30, max_depth=8),
    'no2'       : dict(num_leaves=63,  learning_rate=0.05, min_child_samples=50, max_depth=6),
    'o3'        : dict(num_leaves=127, learning_rate=0.04, min_child_samples=30, max_depth=7),
    'co'        : dict(num_leaves=127, learning_rate=0.04, min_child_samples=30, max_depth=8),
    'so2'       : dict(num_leaves=63,  learning_rate=0.05, min_child_samples=50, max_depth=6),
    'aqi_global': dict(num_leaves=127, learning_rate=0.04, min_child_samples=30, max_depth=8),
}

results_dict['lgbm']    = {}
all_test_preds['lgbm']  = {}
all_valid_preds['lgbm'] = {}
t0_sec = time.time()

for t in TARGETS:
    params = {**LGBM_COMMON, **LGBM_PER[t]}
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train, y_train_log[t],
          eval_set=[(X_valid, y_valid_log[t])],
          callbacks=[lgb.early_stopping(PATIENCE, verbose=False),
                     lgb.log_evaluation(LOG_EVERY)])
    n_trees = getattr(m, 'best_iteration_', m.n_estimators)

    yp_va = inv_transform(m.predict(X_valid), t)
    yp_te = inv_transform(m.predict(X_test),  t)

    metrics_te = evaluate(y_true_test[t], yp_te, t)
    results_dict['lgbm'][t]    = metrics_te
    all_test_preds['lgbm'][t]  = yp_te
    all_valid_preds['lgbm'][t] = yp_va

    model_registry[t]['lgbm'] = {**metrics_te, 'trees': n_trees, '_model_obj': m}
    log.info('LGBM  %-12s  trees=%4d  RMSE=%.4f  R2=%.4f', t, n_trees,
             metrics_te['RMSE'], metrics_te['R2'])

log.info('LightGBM total : %.1f min', (time.time()-t0_sec)/60)
print_table('LightGBM — RMSE test 2025', results_dict, algos=['lgbm'])


20:19:53 [INFO] === Section 3 : LightGBM ===


[200]	valid_0's l2: 0.00482111
[400]	valid_0's l2: 0.00422773
[600]	valid_0's l2: 0.00405275


20:21:03 [INFO] LGBM  pm2_5         trees= 579  RMSE=2.4253  R2=0.9755


[200]	valid_0's l2: 0.00629552
[400]	valid_0's l2: 0.00568105
[600]	valid_0's l2: 0.00550812


20:22:48 [INFO] LGBM  pm10          trees= 598  RMSE=7.5327  R2=0.9717
20:23:01 [INFO] LGBM  no2           trees=  79  RMSE=3.2690  R2=0.5965


...

[200]	valid_0's l2: 0.00145241
[400]	valid_0's l2: 0.00112346
[600]	valid_0's l2: 0.00108862
[800]	valid_0's l2: 0.00107634


20:24:20 [INFO] LGBM  co            trees= 872  RMSE=26.9138  R2=0.9756
20:24:30 [INFO] LGBM  so2           trees= 124  RMSE=0.5259  R2=0.7816


[200]	valid_0's l2: 36.6727


20:24:46 [INFO] LGBM  aqi_global    trees= 312  RMSE=6.0689  R2=0.9655
20:24:46 [INFO] LightGBM total : 4.9 min


  LightGBM — RMSE test 2025  —  RMSE
  Cible                lgbm    Objectif      NB06
  -----------------------------------------------
  pm2_5          ★   2.4253        1.40      1.56
  pm10           ★   7.5327        6.50      6.98
  no2            ★   3.2690        2.50      2.80
  o3             ★   8.7591        7.80      8.31
  co             ★  26.9138       23.00     24.90
  so2            ★   0.5259        0.48      0.51
  aqi_global     ★   6.0689        4.80      5.30


## Section 4 — XGBoost

In [56]:
log.info('=== Section 4 : XGBoost ===')
_dev = 'cuda' if HAS_GPU else 'cpu'
XGB_COMMON = dict(
    n_estimators=3000, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=2.0,
    tree_method='hist', device=_dev,
    random_state=RANDOM_STATE, eval_metric='rmse',
    early_stopping_rounds=PATIENCE,   # ← déplacé ici (XGBoost >= 2.0)
)

# Cibles faciles (R² > 0.95 avec LGBM)
_EASY = ['pm2_5', 'pm10', 'co', 'aqi_global']
# Cibles difficiles (R² < 0.80) : plus de régularisation
_HARD = ['no2', 'so2']
# Cible moyenne : o3
XGB_PER = {
    'pm2_5'     : dict(max_depth=7, min_child_weight=5,  gamma=0.1),
    'pm10'      : dict(max_depth=7, min_child_weight=5,  gamma=0.1),
    'no2'       : dict(max_depth=5, min_child_weight=10, gamma=0.5),
    'o3'        : dict(max_depth=6, min_child_weight=5,  gamma=0.2),
    'co'        : dict(max_depth=7, min_child_weight=5,  gamma=0.1),
    'so2'       : dict(max_depth=5, min_child_weight=10, gamma=0.5),
    'aqi_global': dict(max_depth=7, min_child_weight=5,  gamma=0.1),
}

results_dict['xgboost']    = {}
all_test_preds['xgboost']  = {}
all_valid_preds['xgboost'] = {}

t0_sec = time.time()
for t in TARGETS:
    params = {**XGB_COMMON, **XGB_PER[t]}
    m = xgb.XGBRegressor(**params)
    m.fit(X_train, y_train_log[t],
          eval_set=[(X_valid, y_valid_log[t])],
          verbose=False)                          # ← early_stopping_rounds retiré d'ici
    n_trees = m.best_iteration
    yp_va = inv_transform(m.predict(X_valid), t)
    yp_te = inv_transform(m.predict(X_test),  t)
    metrics_te = evaluate(y_true_test[t], yp_te, t)
    results_dict['xgboost'][t]    = metrics_te
    all_test_preds['xgboost'][t]  = yp_te
    all_valid_preds['xgboost'][t] = yp_va
    model_registry[t]['xgboost'] = {**metrics_te, 'trees': n_trees, '_model_obj': m}
    log.info('XGB   %-12s  trees=%4d  RMSE=%.4f  R2=%.4f', t, n_trees,
             metrics_te['RMSE'], metrics_te['R2'])

log.info('XGBoost total : %.1f min', (time.time()-t0_sec)/60)
print_table('XGBoost — RMSE test 2025', results_dict, algos=['lgbm', 'xgboost'])

20:24:46 [INFO] === Section 4 : XGBoost ===
20:25:03 [INFO] XGB   pm2_5         trees= 335  RMSE=2.5847  R2=0.9721
20:25:20 [INFO] XGB   pm10          trees= 261  RMSE=7.4580  R2=0.9722
20:25:33 [INFO] XGB   no2           trees=  90  RMSE=3.2069  R2=0.6117
20:25:48 [INFO] XGB   o3            trees= 167  RMSE=8.8515  R2=0.8822
20:26:08 [INFO] XGB   co            trees= 519  RMSE=29.6239  R2=0.9704
20:26:13 [INFO] XGB   so2           trees=  62  RMSE=0.5244  R2=0.7829
20:26:51 [INFO] XGB   aqi_global    trees= 569  RMSE=5.9869  R2=0.9664
20:26:51 [INFO] XGBoost total : 2.1 min


  XGBoost — RMSE test 2025  —  RMSE
  Cible                lgbm     xgboost    Objectif      NB06
  -----------------------------------------------------------
  pm2_5          ★   2.4253      2.5847        1.40      1.56
  pm10               7.5327  ★   7.4580        6.50      6.98
  no2                3.2690  ★   3.2069        2.50      2.80
  o3             ★   8.7591      8.8515        7.80      8.31
  co             ★  26.9138     29.6239       23.00     24.90
  so2                0.5259  ★   0.5244        0.48      0.51
  aqi_global         6.0689  ★   5.9869        4.80      5.30


## Section 5 — CatBoost

In [57]:
log.info('=== Section 5 : CatBoost ===')

_task = 'GPU' if HAS_GPU else 'CPU'
CAT_COMMON = dict(
    iterations=3000, learning_rate=0.05,
    border_count=128, random_strength=1,
    loss_function='RMSE', eval_metric='RMSE',
    task_type=_task, random_seed=RANDOM_STATE,
    verbose=LOG_EVERY,
)
CAT_PER = {
    'pm2_5'     : dict(depth=8, l2_leaf_reg=3,  bagging_temperature=0.5),
    'pm10'      : dict(depth=8, l2_leaf_reg=3,  bagging_temperature=0.5),
    'no2'       : dict(depth=6, l2_leaf_reg=10, bagging_temperature=1.0),
    'o3'        : dict(depth=7, l2_leaf_reg=5,  bagging_temperature=0.7),
    'co'        : dict(depth=8, l2_leaf_reg=3,  bagging_temperature=0.5),
    'so2'       : dict(depth=6, l2_leaf_reg=10, bagging_temperature=1.0),
    'aqi_global': dict(depth=8, l2_leaf_reg=3,  bagging_temperature=0.5),
}

results_dict['catboost']    = {}
all_test_preds['catboost']  = {}
all_valid_preds['catboost'] = {}
t0_sec = time.time()

for t in TARGETS:
    params = {**CAT_COMMON, **CAT_PER[t]}
    m = CatBoostRegressor(**params)
    eval_pool = Pool(X_valid, label=y_valid_log[t])
    m.fit(X_train, y_train_log[t],
          eval_set=eval_pool,
          early_stopping_rounds=PATIENCE)
    n_iter = m.get_best_iteration() or m.tree_count_

    yp_va = inv_transform(m.predict(X_valid), t)
    yp_te = inv_transform(m.predict(X_test),  t)

    metrics_te = evaluate(y_true_test[t], yp_te, t)
    results_dict['catboost'][t]    = metrics_te
    all_test_preds['catboost'][t]  = yp_te
    all_valid_preds['catboost'][t] = yp_va

    model_registry[t]['catboost'] = {**metrics_te, 'trees': n_iter, '_model_obj': m}
    log.info('CAT   %-12s  iter=%4d  RMSE=%.4f  R2=%.4f', t, n_iter,
             metrics_te['RMSE'], metrics_te['R2'])

log.info('CatBoost total : %.1f min', (time.time()-t0_sec)/60)
print_table('CatBoost — RMSE test 2025', results_dict, algos=['lgbm', 'xgboost', 'catboost'])


20:26:51 [INFO] === Section 5 : CatBoost ===


0:	learn: 0.7441538	test: 0.6466189	best: 0.6466189 (0)	total: 277ms	remaining: 13m 50s
200:	learn: 0.0758229	test: 0.0745120	best: 0.0745120 (200)	total: 21.4s	remaining: 4m 58s
400:	learn: 0.0516031	test: 0.0677653	best: 0.0677653 (400)	total: 40.5s	remaining: 4m 22s
600:	learn: 0.0424563	test: 0.0629511	best: 0.0629302 (596)	total: 57.6s	remaining: 3m 49s
800:	learn: 0.0372733	test: 0.0604746	best: 0.0604746 (800)	total: 1m 16s	remaining: 3m 31s
1000:	learn: 0.0335789	test: 0.0586972	best: 0.0586741 (995)	total: 1m 40s	remaining: 3m 20s
1200:	learn: 0.0308150	test: 0.0578768	best: 0.0578649 (1189)	total: 1m 59s	remaining: 2m 59s
1400:	learn: 0.0286126	test: 0.0573078	best: 0.0572610 (1395)	total: 2m 17s	remaining: 2m 36s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.05709869359
bestIteration = 1454

Shrink model to first 1455 iterations.


20:29:19 [INFO] CAT   pm2_5         iter=1454  RMSE=2.3874  R2=0.9762


0:	learn: 0.7452603	test: 0.7557110	best: 0.7557110 (0)	total: 121ms	remaining: 6m 3s
200:	learn: 0.0755482	test: 0.0865927	best: 0.0865927 (200)	total: 19.2s	remaining: 4m 26s
400:	learn: 0.0517850	test: 0.0777989	best: 0.0777638 (399)	total: 46.1s	remaining: 4m 58s
600:	learn: 0.0425196	test: 0.0736410	best: 0.0736278 (597)	total: 1m 6s	remaining: 4m 25s
800:	learn: 0.0372682	test: 0.0715207	best: 0.0715207 (800)	total: 1m 26s	remaining: 3m 58s
1000:	learn: 0.0336392	test: 0.0703878	best: 0.0703179 (993)	total: 1m 50s	remaining: 3m 41s
1200:	learn: 0.0309443	test: 0.0689988	best: 0.0689876 (1183)	total: 2m 11s	remaining: 3m 17s
1400:	learn: 0.0287836	test: 0.0682504	best: 0.0682317 (1384)	total: 2m 32s	remaining: 2m 53s
1600:	learn: 0.0269115	test: 0.0677645	best: 0.0677390 (1561)	total: 2m 52s	remaining: 2m 30s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.06773901658
bestIteration = 1561

Shrink model to first 1562 iterations.


20:32:13 [INFO] CAT   pm10          iter=1561  RMSE=7.0605  R2=0.9751


...

0:	learn: 0.7199018	test: 0.4069089	best: 0.4069089 (0)	total: 95.5ms	remaining: 4m 46s
200:	learn: 0.1422559	test: 0.1566895	best: 0.1563236 (196)	total: 13.3s	remaining: 3m 5s


20:35:12 [INFO] CAT   so2           iter= 225  RMSE=0.5162  R2=0.7896


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.1560351981
bestIteration = 225

Shrink model to first 226 iterations.
0:	learn: 46.4064147	test: 33.6325079	best: 33.6325079 (0)	total: 120ms	remaining: 5m 59s
200:	learn: 6.7868828	test: 6.5615262	best: 6.5577069 (192)	total: 25s	remaining: 5m 48s
400:	learn: 5.1597379	test: 6.3519913	best: 6.3461218 (391)	total: 47.4s	remaining: 5m 6s
600:	learn: 4.4554111	test: 6.2389492	best: 6.2389492 (600)	total: 1m 9s	remaining: 4m 37s
800:	learn: 4.0406108	test: 6.1692776	best: 6.1678741 (790)	total: 1m 31s	remaining: 4m 12s
1000:	learn: 3.7472838	test: 6.1075147	best: 6.1075147 (1000)	total: 1m 54s	remaining: 3m 48s
1200:	learn: 3.5139362	test: 6.0646692	best: 6.0624138 (1151)	total: 2m 16s	remaining: 3m 24s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 6.062413772
bestIteration = 1151

Shrink model to first 1152 iterations.


20:37:30 [INFO] CAT   aqi_global    iter=1151  RMSE=6.0388  R2=0.9658
20:37:30 [INFO] CatBoost total : 10.7 min


  CatBoost — RMSE test 2025  —  RMSE
  Cible                lgbm     xgboost    catboost    Objectif      NB06
  -----------------------------------------------------------------------
  pm2_5              2.4253      2.5847  ★   2.3874        1.40      1.56
  pm10               7.5327      7.4580  ★   7.0605        6.50      6.98
  no2                3.2690      3.2069  ★   3.0989        2.50      2.80
  o3             ★   8.7591      8.8515      8.9478        7.80      8.31
  co                26.9138     29.6239  ★  25.8163       23.00     24.90
  so2                0.5259      0.5244  ★   0.5162        0.48      0.51
  aqi_global         6.0689  ★   5.9869      6.0388        4.80      5.30


## Section 6 — Tableau comparatif 3 algorithmes

> ExtraTrees supprimé : trop lourd (~2.5 Go sérialisé) pour un gain négligeable dans l'ensemble.

In [58]:
# ── Section 6 : ExtraTrees supprimé ─────────────────────────────────────────
# ExtraTrees produisait un fichier sérialisé de ~2.5 Go (500 arbres × depth 25)
# pour des performances inférieures à LGBM/XGBoost/CatBoost dans l'ensemble.
# Les algos retenus sont : lgbm, xgboost, catboost (+ lgbm_tuned pour NO2/SO2).

# Initialiser des dicts vides pour 'et' afin de ne pas casser les références ultérieures
results_dict['et']    = {t: {'RMSE': float('inf'), 'MAE': float('inf'),
                              'R2': float('nan'), 'MAPE': float('nan')} for t in TARGETS}
all_test_preds['et']  = {}
all_valid_preds['et'] = {}
for t in TARGETS:
    model_registry[t]['et'] = {}

log.info('ExtraTrees SKIPPED — dicts vides initialisés pour compatibilité')


20:37:30 [INFO] ExtraTrees SKIPPED — dicts vides initialisés pour compatibilité


## Section 7 — Tableau comparatif des 3 algorithmes

In [59]:
ALGOS_3 = ['lgbm', 'xgboost', 'catboost']

print_table('RMSE test 2025 — Comparaison 3 algorithmes', results_dict, 'RMSE', ALGOS_3)
print()
print_table('R2 test 2025 — Comparaison 3 algorithmes',   results_dict, 'R2',   ALGOS_3,
            show_obj=False)

# Vainqueur par cible ──────────────────────────────────────────────────────────
print('\n  VAINQUEUR PAR CIBLE (meilleur RMSE) :')
for t in TARGETS:
    best_a = min(ALGOS_3, key=lambda a: results_dict[a].get(t,{}).get('RMSE', float('inf')))
    best_r = results_dict[best_a][t]['RMSE']
    vs_nb06 = (NB06_RMSE[t] - best_r) / NB06_RMSE[t] * 100
    print(f'  {t:<13}  {best_a:<12}  RMSE={best_r:.4f}  vs NB06={vs_nb06:+.1f}%')


  RMSE test 2025 — Comparaison 3 algorithmes  —  RMSE
  Cible                lgbm     xgboost    catboost    Objectif      NB06
  -----------------------------------------------------------------------
  pm2_5              2.4253      2.5847  ★   2.3874        1.40      1.56
  pm10               7.5327      7.4580  ★   7.0605        6.50      6.98
  no2                3.2690      3.2069  ★   3.0989        2.50      2.80
  o3             ★   8.7591      8.8515      8.9478        7.80      8.31
  co                26.9138     29.6239  ★  25.8163       23.00     24.90
  so2                0.5259      0.5244  ★   0.5162        0.48      0.51
  aqi_global         6.0689  ★   5.9869      6.0388        4.80      5.30

  R2 test 2025 — Comparaison 3 algorithmes  —  R2
  Cible                lgbm     xgboost    catboost
  -------------------------------------------------
  pm2_5              0.9755  ★   0.9721      0.9762
  pm10           ★   0.9717      0.9722      0.9751
  no2            ★   

## Section 8 — Tuning ciblé NO2 & SO2

In [60]:
log.info('=== Section 8 : Features + Tuning NO2/SO2 ===')

# ── 8.1 Features proxy pour sources anthropiques ───────────────────────────────
df2 = df.copy()

# a) Interaction ville × saison sèche (pic NO2 en harmattan pour certaines villes)
for t in ['no2', 'so2']:
    te_col = f'city_te_{t}'
    if te_col in df2.columns:
        df2[f'{t}_city_dry'] = df2[te_col] * df2.get('is_dry_season', 0)

# b) Ratio NO2/CO (proxy type de combustion : moteurs vs feux de biomasse)
if 'no2_lag1' in df2.columns and 'co_lag1' in df2.columns:
    df2['no2_co_ratio_lag1'] = df2['no2_lag1'] / (df2['co_lag1'].clip(lower=1) )

# c) Anomalie hebdomadaire NO2 (si non présente)
if 'no2_weekly_anom' not in df2.columns and 'no2_lag1' in df2.columns:
    if 'no2_roll7_mean' in df2.columns:
        df2['no2_weekly_anom'] = df2['no2_lag1'] - df2['no2_roll7_mean']

# d) Tendance SO2 court terme
if 'so2_lag1' in df2.columns and 'so2_lag3' in df2.columns:
    df2['so2_trend3'] = df2['so2_lag1'] - df2['so2_lag3']

NEW_FEATURES = [c for c in ['no2_city_dry', 'so2_city_dry',
                             'no2_co_ratio_lag1', 'no2_weekly_anom', 'so2_trend3']
                if c in df2.columns]
FEATURE_COLS_V2 = FEATURE_COLS + NEW_FEATURES

# Recalculer les matrices feature
df2['time'] = pd.to_datetime(df2['time'])
df2_train    = df2[df2['time'] <= TRAIN_END]
df2_valid    = df2[(df2['time'] >= VALID_START) & (df2['time'] <= VALID_END)]
df2_test     = df2[df2['time'] >= TEST_START]

X2_train = df2_train[FEATURE_COLS_V2].fillna(0).values.astype(np.float32)
X2_valid = df2_valid[FEATURE_COLS_V2].fillna(0).values.astype(np.float32)
X2_test  = df2_test[FEATURE_COLS_V2].fillna(0).values.astype(np.float32)

log.info('Nouvelles features ajoutées : %s', NEW_FEATURES)
log.info('Nb features v2 : %d', len(FEATURE_COLS_V2))


20:37:30 [INFO] === Section 8 : Features + Tuning NO2/SO2 ===
20:37:31 [INFO] Nouvelles features ajoutées : ['no2_city_dry', 'so2_city_dry', 'no2_co_ratio_lag1', 'no2_weekly_anom', 'so2_trend3']
20:37:31 [INFO] Nb features v2 : 183


In [61]:
# ── 8.2 Tuning LGBM sur NO2 et SO2 ───────────────────────────────────────────
results_dict['lgbm_tuned']    = {t: results_dict['lgbm'][t].copy() for t in TARGETS}
all_test_preds['lgbm_tuned']  = {t: all_test_preds['lgbm'][t].copy() for t in TARGETS}
all_valid_preds['lgbm_tuned'] = {t: all_valid_preds['lgbm'][t].copy() for t in TARGETS}


def _train_lgbm(params, X_tr, y_tr, X_va, y_va, t, X_te=None):
    """
    Entraîne un LGBMRegressor avec early stopping.
    X_te : jeu de test à utiliser pour les prédictions finales.
           Si None, utilise X_test (le jeu de test standard à 178 features).
           Passer X2_test (183 features) quand on entraîne sur X2_train/X2_valid.
    """
    if X_te is None:
        X_te = X_test  # fallback standard
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr, y_tr,
          eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(PATIENCE, verbose=False),
                     lgb.log_evaluation(-1)])
    yp_va = inv_transform(m.predict(X_va), t)
    yp_te = inv_transform(m.predict(X_te), t)
    rmse_va = float(np.sqrt(mean_squared_error(y_true_valid[t], yp_va)))
    return m, yp_va, yp_te, rmse_va


for t_tune in ['no2', 'so2']:
    log.info('--- Tuning %s ---', t_tune)

    if HAS_OPTUNA:
        def objective(trial):
            p = {
                **LGBM_COMMON,
                'num_leaves'       : trial.suggest_int('num_leaves', 31, 255),
                'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
                'reg_alpha'        : trial.suggest_float('reg_alpha', 0.0, 5.0),
                'reg_lambda'       : trial.suggest_float('reg_lambda', 0.0, 10.0),
                'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'n_estimators'     : 3000,
                'max_depth'        : -1,
            }
            _, _, _, rmse_va = _train_lgbm(
                p,
                X2_train, y_train_log[t_tune],
                X2_valid, y_valid_log[t_tune],
                t_tune,
                X_te=X2_test   # ← 183 features, cohérent avec X2_train
            )
            return rmse_va

        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        study.optimize(objective, n_trials=50, timeout=300,
                       show_progress_bar=False)

        # Reconstituer final_params complet depuis les meilleurs params Optuna
        final_params = {
            **LGBM_COMMON,
            **study.best_params,
            'n_estimators': 3000,
            'max_depth'   : -1,
        }
        log.info('%s Optuna best val_RMSE=%.4f | params=%s',
                 t_tune, study.best_value, study.best_params)

    else:
        # Grid search manuel — 3 configs
        configs = {
            'A': {**LGBM_COMMON, **LGBM_PER[t_tune]},
            'B': {**LGBM_COMMON, **LGBM_PER[t_tune],
                  'num_leaves' : max(31, LGBM_PER[t_tune]['num_leaves'] // 2),
                  'reg_lambda' : LGBM_COMMON['reg_lambda'] * 2},
            'C': {**LGBM_COMMON, **LGBM_PER[t_tune],
                  'learning_rate': 0.02},
        }
        final_params, best_va = None, float('inf')
        for cfg_name, cfg in configs.items():
            _, _, _, rmse_va = _train_lgbm(
                cfg,
                X2_train, y_train_log[t_tune],
                X2_valid, y_valid_log[t_tune],
                t_tune,
                X_te=X2_test
            )
            log.info('  Config %s  val_RMSE=%.4f', cfg_name, rmse_va)
            if rmse_va < best_va:
                best_va, final_params = rmse_va, cfg

    # ── Réentraîner avec les meilleurs params et évaluer sur test ──────────────
    m_best, yp_va_best, yp_te_best, _ = _train_lgbm(
        final_params,
        X2_train, y_train_log[t_tune],
        X2_valid, y_valid_log[t_tune],
        t_tune,
        X_te=X2_test   # ← cohérent avec l'entraînement
    )

    metrics_new = evaluate(y_true_test[t_tune], yp_te_best, t_tune)
    metrics_old = results_dict['lgbm'][t_tune]
    gain = (metrics_old['RMSE'] - metrics_new['RMSE']) / metrics_old['RMSE'] * 100

    results_dict['lgbm_tuned'][t_tune]    = metrics_new
    all_test_preds['lgbm_tuned'][t_tune]  = yp_te_best
    all_valid_preds['lgbm_tuned'][t_tune] = yp_va_best
    model_registry[t_tune]['lgbm_tuned']  = {**metrics_new, '_model_obj': m_best}  # ← plus de 'mp'

    log.info('%s  RMSE avant=%.4f  apres=%.4f  gain=%+.1f%%',
             t_tune, metrics_old['RMSE'], metrics_new['RMSE'], gain)

    log.info('%s  RMSE avant=%.4f  apres=%.4f  gain=%+.1f%%',
             t_tune, metrics_old['RMSE'], metrics_new['RMSE'], gain)

print_table('LGBM Tuné vs LGBM Base — NO2 & SO2',
            results_dict, algos=['lgbm', 'lgbm_tuned'], show_obj=False)

20:37:31 [INFO] --- Tuning no2 ---
20:42:35 [INFO] no2 Optuna best val_RMSE=2.9661 | params={'num_leaves': 38, 'learning_rate': 0.08115595675970502, 'min_child_samples': 40, 'reg_alpha': 3.31261142176991, 'reg_lambda': 3.1171107608941098, 'subsample': 0.8080272084711243, 'colsample_bytree': 0.8186841117373118}
20:42:41 [INFO] no2  RMSE avant=3.2690  apres=3.0855  gain=+5.6%
20:42:41 [INFO] no2  RMSE avant=3.2690  apres=3.0855  gain=+5.6%
20:42:41 [INFO] --- Tuning so2 ---
20:47:56 [INFO] so2 Optuna best val_RMSE=0.5311 | params={'num_leaves': 174, 'learning_rate': 0.010139048090380883, 'min_child_samples': 98, 'reg_alpha': 1.2481266542059481, 'reg_lambda': 0.14540875162364375, 'subsample': 0.9630659181130071, 'colsample_bytree': 0.8921467580828037}
20:49:06 [INFO] so2  RMSE avant=0.5259  apres=0.5404  gain=-2.8%
20:49:06 [INFO] so2  RMSE avant=0.5259  apres=0.5404  gain=-2.8%


  LGBM Tuné vs LGBM Base — NO2 & SO2  —  RMSE
  Cible                lgbm  lgbm_tuned
  -------------------------------------
  pm2_5          ★   2.4253      2.4253
  pm10           ★   7.5327      7.5327
  no2                3.2690  ★   3.0855
  o3             ★   8.7591      8.7591
  co             ★  26.9138     26.9138
  so2            ★   0.5259      0.5404
  aqi_global     ★   6.0689      6.0689


## Section 9 — Ensemble final

In [62]:
log.info('=== Section 9 : Ensemble ===')

# Algos disponibles pour l'ensemble (exclure baselines)
ENSEMBLE_ALGOS = ['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']

# ── 9.1 & 9.2 — Ensemble pondéré (w = 1/RMSE_valid²) ─────────────────────────
results_dict['ensemble_w']    = {}
all_test_preds['ensemble_w']  = {}
all_valid_preds['ensemble_w'] = {}

for t in TARGETS:
    # Calculer RMSE sur validation pour chaque algo
    weights = {}
    for a in ENSEMBLE_ALGOS:
        if t not in all_valid_preds.get(a, {}):
            continue
        rmse_va = float(np.sqrt(mean_squared_error(
            y_true_valid[t], all_valid_preds[a][t])))
        # Exclure les modèles dont RMSE_test > BL × 1.2
        bl_rmse = results_dict['bl_persistence'][t]['RMSE']
        if rmse_va < bl_rmse * 1.2:
            weights[a] = 1.0 / (rmse_va ** 2 + 1e-8)

    if not weights:
        weights = {'lgbm': 1.0}

    total_w = sum(weights.values())
    norm_w  = {a: w / total_w for a, w in weights.items()}

    # Ensemble test
    yp_te = sum(norm_w[a] * all_test_preds[a][t]
                for a in norm_w if t in all_test_preds.get(a, {}))
    yp_va = sum(norm_w[a] * all_valid_preds[a][t]
                for a in norm_w if t in all_valid_preds.get(a, {}))

    results_dict['ensemble_w'][t]    = evaluate(y_true_test[t], yp_te, t)
    all_test_preds['ensemble_w'][t]  = yp_te
    all_valid_preds['ensemble_w'][t] = yp_va

    log.info('EnsW %-12s  algos=%s  RMSE=%.4f',
             t, list(norm_w.keys()), results_dict['ensemble_w'][t]['RMSE'])

# ── 9.3 — Stacking Ridge ──────────────────────────────────────────────────────
results_dict['stacking']    = {}
all_test_preds['stacking']  = {}
all_valid_preds['stacking'] = {}

for t in TARGETS:
    avail_algos = [a for a in ENSEMBLE_ALGOS
                   if t in all_valid_preds.get(a, {}) and t in all_test_preds.get(a, {})]
    if len(avail_algos) < 2:
        results_dict['stacking'][t]    = results_dict['lgbm'][t]
        all_test_preds['stacking'][t]  = all_test_preds['lgbm'][t]
        all_valid_preds['stacking'][t] = all_valid_preds['lgbm'][t]
        continue

    Xm_va = np.column_stack([all_valid_preds[a][t] for a in avail_algos])
    Xm_te = np.column_stack([all_test_preds[a][t]  for a in avail_algos])
    meta  = Ridge(alpha=10.0, positive=True)
    meta.fit(Xm_va, y_true_valid[t])
    yp_va = np.clip(meta.predict(Xm_va), 0, None)
    yp_te = np.clip(meta.predict(Xm_te), 0, None)

    results_dict['stacking'][t]    = evaluate(y_true_test[t], yp_te, t)
    all_test_preds['stacking'][t]  = yp_te
    all_valid_preds['stacking'][t] = yp_va

# ── 9.4 — Correction de biais par ville ───────────────────────────────────────
results_dict['stacking_bc']    = {}
all_test_preds['stacking_bc']  = {}
all_valid_preds['stacking_bc'] = {}

for t in TARGETS:
    yp_va = all_valid_preds['stacking'][t]
    resid  = y_true_valid[t] - yp_va
    cb     = pd.Series(resid).groupby(df_valid['city_indabax'].values).median()

    corr = df_test['city_indabax'].map(cb).fillna(0).values
    yp_te_bc = np.clip(all_test_preds['stacking'][t] + corr, 0, None)

    m_bc = evaluate(y_true_test[t], yp_te_bc, t)
    m_st = results_dict['stacking'][t]

    # Ne corriger que si gain > 2%
    if m_bc['RMSE'] < m_st['RMSE'] * 0.98:
        results_dict['stacking_bc'][t]    = m_bc
        all_test_preds['stacking_bc'][t]  = yp_te_bc
        all_valid_preds['stacking_bc'][t] = yp_va
        log.info('BiasCorr %-12s  RMSE %.4f → %.4f  gain=+%.1f%%',
                 t, m_st['RMSE'], m_bc['RMSE'],
                 (m_st['RMSE']-m_bc['RMSE'])/m_st['RMSE']*100)
    else:
        results_dict['stacking_bc'][t]    = m_st
        all_test_preds['stacking_bc'][t]  = all_test_preds['stacking'][t]
        all_valid_preds['stacking_bc'][t] = all_valid_preds['stacking'][t]

print_table('Ensemble — RMSE test', results_dict,
            algos=['lgbm', 'ensemble_w', 'stacking', 'stacking_bc'])


20:49:06 [INFO] === Section 9 : Ensemble ===
20:49:06 [INFO] EnsW pm2_5         algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=2.3601
20:49:06 [INFO] EnsW pm10          algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=7.2223
20:49:06 [INFO] EnsW no2           algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=3.1384
20:49:06 [INFO] EnsW o3            algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=8.7462
20:49:06 [INFO] EnsW co            algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=25.1450
20:49:06 [INFO] EnsW so2           algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=0.5197
20:49:06 [INFO] EnsW aqi_global    algos=['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']  RMSE=5.8918
20:49:06 [INFO] BiasCorr pm2_5         RMSE 1.5733 → 1.4757  gain=+6.2%
20:49:06 [INFO] BiasCorr pm10          RMSE 6.9977 → 6.4645  gain=+7.6%
20:49:06 [INFO] BiasCorr no2           RMSE 2.7256 → 2.6536  gain=+2.6%
20:49:06 [INFO] BiasCorr co           

  Ensemble — RMSE test  —  RMSE
  Cible                lgbm  ensemble_w    stacking  stacking_b    Objectif      NB06
  -----------------------------------------------------------------------------------
  pm2_5              2.4253      2.3601      1.5733  ★   1.4757        1.40      1.56
  pm10               7.5327      7.2223      6.9977  ★   6.4645        6.50      6.98
  no2                3.2690      3.1384      2.7256  ★   2.6536        2.50      2.80
  o3                 8.7591      8.7462  ★   8.3294      8.3294        7.80      8.31
  co                26.9138     25.1450     23.0198  ★  22.0061       23.00     24.90
  so2                0.5259      0.5197  ★   0.5051      0.5051        0.48      0.51
  aqi_global         6.0689      5.8918      5.3697  ★   5.1485        4.80      5.30


## Section 10 — Sélection finale par cible

In [63]:
log.info('=== Section 10 : Selection finale ===')

ALL_VARIANTS = (ENSEMBLE_ALGOS +
                ['ensemble_w', 'stacking', 'stacking_bc'])

final_algo   = {}   # {target: algo_name}
final_preds  = {}   # {target: np.array test}
final_valid  = {}   # {target: np.array valid}
final_metrics = {}  # {target: metrics_dict}

for t in TARGETS:
    best_a, best_r = None, float('inf')
    for a in ALL_VARIANTS:
        r = results_dict.get(a, {}).get(t, {}).get('RMSE', float('inf'))
        if r < best_r:
            best_r, best_a = r, a

    final_algo[t]    = best_a
    final_preds[t]   = all_test_preds[best_a][t]
    final_valid[t]   = all_valid_preds.get(best_a, all_valid_preds['lgbm'])[t]
    final_metrics[t] = results_dict[best_a][t]

# Tableau final vs NB06
print()
print('=' * 95)
print('  SELECTION FINALE — RMSE test 2025')
print('=' * 95)
print(f'  {"Cible":<13}  {"Methode":>14}  {"RMSE":>8}  {"MAE":>8}  {"R2":>7}  '
      f'{"MAPE%":>8}  {"NB06":>8}  {"Gain%":>8}  {"Objectif":>10}  {"OK?":>5}')
print('  ' + '-' * 90)
all_ok = True
for t in TARGETS:
    m    = final_metrics[t]
    nb06 = NB06_RMSE[t]
    obj  = OBJ_RMSE[t]
    gain = (nb06 - m['RMSE']) / nb06 * 100
    ok   = 'OK' if m['RMSE'] <= obj else 'MISS'
    if ok == 'MISS':
        all_ok = False
    mape_s = f'{m["MAPE"]:>7.1f}%' if not np.isnan(m['MAPE']) else '    N/A '
    print(f'  {t:<13}  {final_algo[t]:>14}  {m["RMSE"]:>8.4f}  {m["MAE"]:>8.4f}  '
          f'{m["R2"]:>7.4f}  {mape_s}  {nb06:>8.2f}  {gain:>+7.1f}%  '
          f'{obj:>10.2f}  {ok:>5}')
print('=' * 95)
print(f'  Objectifs atteints : {sum(1 for t in TARGETS if final_metrics[t]["RMSE"] <= OBJ_RMSE[t])}/{len(TARGETS)}')


20:49:06 [INFO] === Section 10 : Selection finale ===



  SELECTION FINALE — RMSE test 2025
  Cible                 Methode      RMSE       MAE       R2     MAPE%      NB06     Gain%    Objectif    OK?
  ------------------------------------------------------------------------------------------
  pm2_5             stacking_bc    1.4757    0.9695   0.9909      5.4%      1.56     +5.4%        1.40   MISS
  pm10              stacking_bc    6.4645    3.6155   0.9791     11.1%      6.98     +7.4%        6.50     OK
  no2               stacking_bc    2.6536    1.8384   0.7341     29.9%      2.80     +5.2%        2.50   MISS
  o3                   stacking    8.3294    5.5773   0.8957      7.0%      8.31     -0.2%        7.80   MISS
  co                stacking_bc   22.0061   10.9699   0.9837      2.5%     24.90    +11.6%       23.00     OK
  so2                  stacking    0.5051    0.3106   0.7986     28.9%      0.51     +1.0%        0.48   MISS
  aqi_global        stacking_bc    5.1485    2.9166   0.9751      4.4%      5.30     +2.9%        4.

## Section 11 — Analyse overfitting

In [64]:
log.info('=== Section 11 : Verification overfitting ===')

print('=' * 65)
print('  RATIO RMSE_TRAIN / RMSE_TEST')
print('  0.8-1.3 = OK  |  > 1.3 = Overfitting  |  < 0.8 = Sous-apprentissage')
print('=' * 65)

retrain_needed = []
for t in TARGETS:
    a = final_algo[t]
    a_base = 'lgbm' if a in ('ensemble_w', 'stacking', 'stacking_bc') else a

    # Récupérer l'objet modèle depuis model_registry (_model_obj)
    m_obj = model_registry[t].get(a_base, {}).get('_model_obj')

    if m_obj is not None:
        fc = FEATURE_COLS_V2 if a_base == 'lgbm_tuned' else FEATURE_COLS
        yp_tr_log = m_obj.predict(X_train[:, :len(fc)] if X_train.shape[1] > len(fc) else X_train)
        yp_tr     = inv_transform(yp_tr_log, t)
        rmse_tr   = float(np.sqrt(mean_squared_error(y_true_train[t], yp_tr)))
    else:
        rmse_tr = final_metrics[t]['RMSE'] * 0.9   # proxy si indisponible

    rmse_te = final_metrics[t]['RMSE']
    ratio   = rmse_tr / rmse_te if rmse_te > 0 else float('nan')
    flag    = ('OK'       if 0.8 <= ratio <= 1.3 else
               'OVERFIT'  if ratio > 1.3          else 'SOUS-APP')
    print(f'  {t:<13}  train={rmse_tr:>8.4f}  test={rmse_te:>8.4f}  '
          f'ratio={ratio:>6.3f}  {flag}')
    if flag == 'OVERFIT':
        retrain_needed.append(t)

print('=' * 65)
if retrain_needed:
    print(f'  Alertes overfitting : {retrain_needed}')
    print('  Action : augmenter reg_lambda x2, réduire num_leaves/depth')
else:
    print('  Aucun overfitting détecté.')


20:49:06 [INFO] === Section 11 : Verification overfitting ===


  RATIO RMSE_TRAIN / RMSE_TEST
  0.8-1.3 = OK  |  > 1.3 = Overfitting  |  < 0.8 = Sous-apprentissage
  pm2_5          train=  1.8915  test=  1.4757  ratio= 1.282  OK
  pm10           train=  2.6360  test=  6.4645  ratio= 0.408  SOUS-APP
  no2            train=  2.1264  test=  2.6536  ratio= 0.801  OK
  o3             train=  5.3551  test=  8.3294  ratio= 0.643  SOUS-APP
  co             train= 19.4982  test= 22.0061  ratio= 0.886  OK
  so2            train=  0.8323  test=  0.5051  ratio= 1.648  OVERFIT
  aqi_global     train=  3.8947  test=  5.1485  ratio= 0.756  SOUS-APP
  Alertes overfitting : ['so2']
  Action : augmenter reg_lambda x2, réduire num_leaves/depth


## Section 12 — Feature Importance (top 15 par cible)

In [65]:
log.info('=== Section 12 : Feature importance ===')

# Cohérence physique attendue
EXPECTED_TOP5 = {
    'pm2_5'     : ['pm2_5_lag1', 'is_harmattan', 'precipitation_sum'],
    'pm10'      : ['pm10_lag1',  'is_harmattan', 'wind_speed_10m_max'],
    'no2'       : ['city_te_no2', 'city_te_pm2_5', 'eco_zone_le'],
    'o3'        : ['sunshine_fraction', 'temperature_2m_mean', 'shortwave_radiation_sum'],
    'co'        : ['co_lag1', 'is_harmattan', 'pm2_5_lag1'],
    'so2'       : ['city_te_so2', 'eco_zone_le', 'so2_lag1'],
    'aqi_global': ['pm2_5_lag1', 'pm2_5_roll7_mean', 'aqi_global_lag1'],
}

fig, axes = plt.subplots(3, 3, figsize=(20, 18))
axes_flat = axes.flatten()

for idx, t in enumerate(TARGETS):
    ax = axes_flat[idx]
    a  = final_algo[t]
    # Choisir le modèle de base pour l'importance
    if a in ('ensemble_w', 'stacking', 'stacking_bc'):
        a_base = 'lgbm'
    else:
        a_base = a

    m_obj = model_registry[t].get(a_base, {}).get('_model_obj')
    if m_obj is None:
        ax.set_visible(False)
        continue
    fc = FEATURE_COLS_V2 if a_base == 'lgbm_tuned' else FEATURE_COLS

    try:
        if a_base in ('lgbm', 'lgbm_tuned'):
            imp = m_obj.feature_importances_
        elif a_base == 'xgboost':
            imp = m_obj.feature_importances_
        elif a_base == 'catboost':
            imp = m_obj.get_feature_importance()
    
        else:
            ax.set_visible(False)
            continue

        imp_s = pd.Series(imp, index=fc[:len(imp)]).nlargest(15)
        colors = ['#e74c3c' if f in EXPECTED_TOP5.get(t, []) else '#3498db'
                  for f in imp_s.index]
        imp_s.plot(kind='barh', ax=ax, color=colors[::-1])
        ax.invert_yaxis()
        ax.set_title(f'{t.upper()} — top 15 ({a_base})', fontsize=10)
        ax.set_xlabel('Importance')
        ax.tick_params(axis='y', labelsize=7)

        # Vérification cohérence physique
        top5 = set(imp_s.head(5).index)
        expected = set(EXPECTED_TOP5.get(t, []))
        n_match = len(top5 & expected)
        ax.text(0.99, 0.02, f'Cohérence: {n_match}/{len(expected)}',
                transform=ax.transAxes, ha='right', fontsize=8, color='gray')
    except Exception as e:
        ax.text(0.5, 0.5, str(e), transform=ax.transAxes, ha='center')

# Dernier subplot : légende
axes_flat[7].set_visible(False)
axes_flat[8].set_visible(False)
from matplotlib.patches import Patch
leg_ax = fig.add_subplot(3, 3, 9)
leg_ax.legend(handles=[Patch(facecolor='#e74c3c', label='Feature attendue en top 5'),
                        Patch(facecolor='#3498db', label='Autre feature')],
              loc='center', fontsize=11)
leg_ax.axis('off')

plt.suptitle('Feature Importance — Top 15 par cible', fontsize=14, y=1.01)
plt.tight_layout()
out_fi = DATA_OUT / 'nb08_feature_importance.png'
plt.savefig(out_fi, dpi=120, bbox_inches='tight')
plt.show()
log.info('Feature importance sauvegardée : %s', out_fi)

# Alerte data-drift si days_since_start domine
for t in TARGETS:
    a_base = 'lgbm' if final_algo[t] in ('ensemble_w','stacking','stacking_bc') else final_algo[t]
    m_obj  = model_registry[t].get(a_base, {}).get('_model_obj')
    if m_obj is not None:
        fc_use = FEATURE_COLS_V2 if a_base == 'lgbm_tuned' else FEATURE_COLS
        try:
            imp  = pd.Series(m_obj.feature_importances_, index=fc_use[:len(m_obj.feature_importances_)])
            top3 = imp.nlargest(3).index.tolist()
            if 'days_since_start' in top3:
                log.warning('DATA DRIFT POTENTIEL sur %s : days_since_start en top3', t)
        except Exception:
            pass


20:49:22 [INFO] === Section 12 : Feature importance ===
20:49:27 [INFO] Feature importance sauvegardée : C:\Users\NICK-TECH\OneDrive\Desktop\Deepstsat\models\nb08_feature_importance.png
20:49:27 [WARNING] DATA DRIFT POTENTIEL sur pm2_5 : days_since_start en top3
20:49:27 [WARNING] DATA DRIFT POTENTIEL sur pm10 : days_since_start en top3
20:49:27 [WARNING] DATA DRIFT POTENTIEL sur no2 : days_since_start en top3
20:49:27 [WARNING] DATA DRIFT POTENTIEL sur co : days_since_start en top3
20:49:27 [WARNING] DATA DRIFT POTENTIEL sur so2 : days_since_start en top3
20:49:27 [WARNING] DATA DRIFT POTENTIEL sur aqi_global : days_since_start en top3


## Section 13 — Sauvegarde & Export

In [66]:
log.info('=== Section 13 : Sauvegarde PKL optimisée ===')

import pickle, gzip

# ── 13.0  Récupération des objets atomiques ────────────────────────────────────
atomic_model_objects = {}
for t in TARGETS:
    for a in ['lgbm', 'xgboost', 'catboost', 'lgbm_tuned']:
        obj = model_registry[t].get(a, {}).get('_model_obj')
        if obj is not None:
            atomic_model_objects.setdefault(a, {})[t] = obj

log.info('Objets atomiques récupérés : %s',
         {a: list(d.keys()) for a, d in atomic_model_objects.items()})

# ── 13.1  Sauvegarde UNIQUE des modèles atomiques (shared_models.pkl.gz) ──────
# Un seul fichier compressé contient tous les modèles de base.
# Les PKL par cible référencent ce fichier par chemin (pas de duplication).
shared_payload = {
    'models'          : atomic_model_objects,   # {algo: {target: model_obj}}
    'feature_cols'    : FEATURE_COLS,
    'feature_cols_v2' : FEATURE_COLS_V2,
    'targets'         : TARGETS,
    'trained_on'      : f'2020-01-01/{TRAIN_END}',
}
shared_pkl = MODELS_DIR / 'shared_models.pkl.gz'
with gzip.open(shared_pkl, 'wb', compresslevel=3) as fh:
    pickle.dump(shared_payload, fh, protocol=4)
log.info('shared_models.pkl.gz : %.1f Mo', shared_pkl.stat().st_size / 1e6)

# ── 13.2  Reconstruction des meta-modèles stacking ────────────────────────────
stacking_meta_models = {}
stacking_bias_corr   = {}

for t in TARGETS:
    avail_algos = [a for a in ENSEMBLE_ALGOS
                   if t in all_valid_preds.get(a, {}) and t in all_test_preds.get(a, {})]
    if len(avail_algos) >= 2:
        Xm_va = np.column_stack([all_valid_preds[a][t] for a in avail_algos])
        meta  = Ridge(alpha=10.0, positive=True)
        meta.fit(Xm_va, y_true_valid[t])
        stacking_meta_models[t] = {'meta': meta, 'algos': avail_algos}

        yp_va = np.clip(meta.predict(Xm_va), 0, None)
        resid = y_true_valid[t] - yp_va
        cb    = pd.Series(resid).groupby(df_valid['city_indabax'].values).median()
        stacking_bias_corr[t] = cb

log.info('Meta-modèles Ridge reconstruits pour : %s', list(stacking_meta_models.keys()))

# ── 13.3  PKL léger par cible (meta-model + biais uniquement) ─────────────────
pkl_paths   = {}
live_models = {}

for t in TARGETS:
    a      = final_algo[t]
    fc_use = FEATURE_COLS_V2 if a == 'lgbm_tuned' else FEATURE_COLS

    # ── Cas 1 : modèle atomique ───────────────────────────────────────────────
    if a in ('lgbm', 'xgboost', 'catboost', 'lgbm_tuned'):
        payload = {
            'type'             : 'atomic',
            'algo'             : a,
            'shared_models_ref': str(shared_pkl),
            'features'         : fc_use,
            'target'           : t,
            'trained_on'       : f'2020-01-01/{TRAIN_END}',
        }

    # ── Cas 2 : stacking / stacking_bc ────────────────────────────────────────
    elif a in ('stacking', 'stacking_bc') and t in stacking_meta_models:
        info = stacking_meta_models[t]
        payload = {
            'type'             : a,
            'algo'             : a,
            'meta_model'       : info['meta'],          # Ridge (quelques Ko)
            'base_algos'       : info['algos'],         # liste de strings
            'shared_models_ref': str(shared_pkl),
            'bias_corr'        : stacking_bias_corr.get(t) if a == 'stacking_bc' else None,
            'features'         : fc_use,
            'target'           : t,
            'trained_on'       : f'2020-01-01/{TRAIN_END}',
        }

    # ── Cas 3 : ensemble_w ────────────────────────────────────────────────────
    elif a == 'ensemble_w':
        weights = {}
        for ea in ENSEMBLE_ALGOS:
            if t not in all_valid_preds.get(ea, {}):
                continue
            rmse_va = float(np.sqrt(mean_squared_error(
                y_true_valid[t], all_valid_preds[ea][t])))
            bl_rmse = results_dict['bl_persistence'][t]['RMSE']
            if rmse_va < bl_rmse * 1.2:
                weights[ea] = 1.0 / (rmse_va ** 2 + 1e-8)
        total_w = sum(weights.values()) or 1.0
        norm_w  = {ea: w / total_w for ea, w in weights.items()}
        payload = {
            'type'             : 'ensemble_w',
            'algo'             : 'ensemble_w',
            'weights'          : norm_w,
            'base_algos'       : list(norm_w.keys()),
            'shared_models_ref': str(shared_pkl),
            'features'         : fc_use,
            'target'           : t,
            'trained_on'       : f'2020-01-01/{TRAIN_END}',
        }
    else:
        log.warning('Type non géré pour %s (%s) — skip PKL', t, a)
        continue

    pkl_path = MODELS_DIR / f'{t}_best_{a}.pkl'
    with open(pkl_path, 'wb') as fh:
        pickle.dump(payload, fh, protocol=4)
    pkl_paths[t]   = pkl_path
    live_models[t] = payload
    log.info('PKL %-13s  %-14s  %.2f Mo', t, a,
             pkl_path.stat().st_size / 1e6)

# ── 13.4  Registre JSON ────────────────────────────────────────────────────────
registry_export = {}
for t in TARGETS:
    fc_use = FEATURE_COLS_V2 if final_algo[t] == 'lgbm_tuned' else FEATURE_COLS
    registry_export[t] = {
        'algo'           : final_algo[t],
        'type'           : live_models.get(t, {}).get('type', 'unknown'),
        'rmse_test'      : float(final_metrics[t]['RMSE']),
        'mae_test'       : float(final_metrics[t]['MAE']),
        'r2_test'        : float(final_metrics[t]['R2']),
        'mape_test'      : float(final_metrics[t]['MAPE']) if not np.isnan(final_metrics[t]['MAPE']) else None,
        'n_features'     : len(fc_use),
        'features'       : fc_use[:20],
        'trained_on'     : f'2020-01-01/{TRAIN_END}',
        'pkl_file'       : str(pkl_paths.get(t, 'N/A')),
        'shared_models'  : str(shared_pkl),
    }

reg_path = DATA_OUT / 'model_registry.json'
with open(reg_path, 'w', encoding='utf-8') as fh:
    json.dump(registry_export, fh, indent=2, ensure_ascii=False)
log.info('model_registry.json : %s', reg_path)

# ── 13.5  Résumé des tailles ──────────────────────────────────────────────────
print('\n' + '=' * 55)
print('  FICHIERS PRODUITS')
print('=' * 55)
print(f'  {"Fichier":<38} {"Taille":>8}')
print('  ' + '-' * 50)
print(f'  {"shared_models.pkl.gz":<38} {shared_pkl.stat().st_size/1e6:>7.1f} Mo')
for t in TARGETS:
    fp = pkl_paths.get(t)
    if fp and fp.exists():
        print(f'  {fp.name:<38} {fp.stat().st_size/1e6:>7.3f} Mo')
if reg_path.exists():
    print(f'  {"model_registry.json":<38} {reg_path.stat().st_size/1e6:>7.3f} Mo')
print('  ' + '-' * 50)
total = shared_pkl.stat().st_size + sum(
    pkl_paths[t].stat().st_size for t in TARGETS
    if t in pkl_paths and pkl_paths[t].exists())
print(f'  {"TOTAL":<38} {total/1e6:>7.1f} Mo')
print('=' * 55)

# ── 13.6  Analyse erreurs par ville et par saison ─────────────────────────────
pred_df = df_test[['city_indabax', 'time', 'region_indabax', 'month', 'season']].copy()
for t in TARGETS:
    pred_df[f'y_true_{t}'] = y_true_test[t]
    pred_df[f'y_pred_{t}'] = np.clip(final_preds[t], 0, None)
    pred_df[f'erreur_{t}'] = pred_df[f'y_pred_{t}'] - pred_df[f'y_true_{t}']
pred_df = pred_df.sort_values(['city_indabax', 'time']).reset_index(drop=True)

for t_show in ['pm2_5', 'aqi_global']:
    err_df = pred_df[['city_indabax', 'region_indabax', 'season',
                       f'y_true_{t_show}', f'y_pred_{t_show}',
                       f'erreur_{t_show}']].copy()
    err_df['abs_err'] = np.abs(err_df[f'erreur_{t_show}'])

    city_rmse = (err_df.groupby('city_indabax').apply(
        lambda g: np.sqrt(mean_squared_error(g[f'y_true_{t_show}'], g[f'y_pred_{t_show}'])))
                 .nlargest(10))
    print(f'\n  Top 10 villes difficiles — {t_show.upper()}')
    for city, rmse in city_rmse.items():
        print(f'    {city:<20} RMSE={rmse:.3f}')

    print(f'\n  RMSE par saison — {t_show.upper()}')
    for s, g in err_df.groupby('season'):
        r = np.sqrt(mean_squared_error(g[f'y_true_{t_show}'], g[f'y_pred_{t_show}']))
        print(f'    {str(s):<20} RMSE={r:.3f}  (n={len(g)})')

    worst = err_df.nlargest(10, 'abs_err')[
        ['city_indabax', f'y_true_{t_show}', f'y_pred_{t_show}', 'abs_err']]
    print(f'\n  10 pires prédictions — {t_show.upper()}')
    print(worst.to_string(index=False))


20:49:27 [INFO] === Section 13 : Sauvegarde PKL optimisée ===
20:49:27 [INFO] Objets atomiques récupérés : {'lgbm': ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'aqi_global'], 'xgboost': ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'aqi_global'], 'catboost': ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'aqi_global'], 'lgbm_tuned': ['no2', 'so2']}
20:49:32 [INFO] shared_models.pkl.gz : 25.9 Mo
20:49:33 [INFO] Meta-modèles Ridge reconstruits pour : ['pm2_5', 'pm10', 'no2', 'o3', 'co', 'so2', 'aqi_global']
20:49:33 [INFO] PKL pm2_5          stacking_bc     0.00 Mo
20:49:33 [INFO] PKL pm10           stacking_bc     0.00 Mo
20:49:33 [INFO] PKL no2            stacking_bc     0.00 Mo
20:49:33 [INFO] PKL o3             stacking        0.00 Mo
20:49:33 [INFO] PKL co             stacking_bc     0.00 Mo
20:49:33 [INFO] PKL so2            stacking        0.00 Mo
20:49:33 [INFO] PKL aqi_global     stacking_bc     0.00 Mo
20:49:33 [INFO] model_registry.json : C:\Users\NICK-TECH\OneDrive\Desktop\Deepsts


  FICHIERS PRODUITS
  Fichier                                  Taille
  --------------------------------------------------
  shared_models.pkl.gz                      25.9 Mo
  pm2_5_best_stacking_bc.pkl               0.005 Mo
  pm10_best_stacking_bc.pkl                0.005 Mo
  no2_best_stacking_bc.pkl                 0.005 Mo
  o3_best_stacking.pkl                     0.004 Mo
  co_best_stacking_bc.pkl                  0.005 Mo
  so2_best_stacking.pkl                    0.004 Mo
  aqi_global_best_stacking_bc.pkl          0.005 Mo
  model_registry.json                      0.008 Mo
  --------------------------------------------------
  TOTAL                                     25.9 Mo

  Top 10 villes difficiles — PM2_5
    Mokolo               RMSE=2.633
    Foumban              RMSE=2.612
    Batouri              RMSE=2.298
    Meiganga             RMSE=2.149
    Bertoua              RMSE=1.997
    Touboro              RMSE=1.874
    Kousseri             RMSE=1.790
    Yokadouma  

## Section 14 — Rapport Final

In [67]:
n_ok   = sum(1 for t in TARGETS if final_metrics[t]['RMSE'] <= OBJ_RMSE[t])
n_beat = sum(1 for t in TARGETS if final_metrics[t]['RMSE'] < NB06_RMSE[t])

print()
print('=' * 70)
print('  RAPPORT FINAL — NOTEBOOK 08 Multi-algorithmes')
print('  IndabaX Cameroon 2026')
print('=' * 70)
print()
print('  1. CONFIGURATION')
print(f'     Train      : 2020-01-01 → {TRAIN_END}  ({len(df_train):,} lignes)')
print(f'     Validation : {VALID_START} → {VALID_END}  ({len(df_valid):,} lignes)')
print(f'     Test       : {TEST_START} →   ({len(df_test):,} lignes)')
print(f'     Features   : {len(FEATURE_COLS)} (base) + {len(NEW_FEATURES)} (NO2/SO2)')
print(f'     Algorithmes: LGBM, XGBoost, CatBoost + Tuning NO2/SO2 + Ensemble')
print()

print('  2. TABLEAU DE PERFORMANCES FINAL')
print(f'  {"Cible":<13}  {"Algo retenu":>18}  {"RMSE":>8}  {"R2":>7}  {"vs NB06":>9}  {"Objectif":>10}  {"OK?":>5}')
print('  ' + '-' * 76)
for t in TARGETS:
    m    = final_metrics[t]
    gain = (NB06_RMSE[t] - m['RMSE']) / NB06_RMSE[t] * 100
    ok   = 'OK' if m['RMSE'] <= OBJ_RMSE[t] else '--'
    print(f'  {t:<13}  {final_algo[t]:>18}  {m["RMSE"]:>8.4f}  {m["R2"]:>7.4f}  '
          f'{gain:>+8.1f}%  {OBJ_RMSE[t]:>10.2f}  {ok:>5}')
print()

print('  3. VAINQUEUR PAR CIBLE')
for t in TARGETS:
    m = final_metrics[t]
    print(f'     {t:<13} → {final_algo[t]}  (RMSE={m["RMSE"]:.4f}  R²={m["R2"]:.4f})')
print()

print(f'  4. ROBUSTESSE')
print(f'     Objectifs atteints   : {n_ok}/{len(TARGETS)}')
print(f'     Cibles battant NB06  : {n_beat}/{len(TARGETS)}')
print()

print('  5. RECOMMANDATION FINALE')
if n_beat >= 5:
    best_t = min(TARGETS, key=lambda t: final_metrics[t]['RMSE'] / NB06_RMSE[t])
    print(f'     Le modele NB08 (ensemble multi-algo) améliore {n_beat}/7 cibles.')
    print(f'     Plus grand gain : {best_t} avec {final_algo[best_t]}.')
    print('     RECOMMANDATION : Utiliser NB08 comme soumission principale.')
else:
    print(f'     Seulement {n_beat}/7 cibles améliorées vs NB06.')
    print('     RECOMMANDATION : Garder NB06 (LightGBM) pour les cibles non améliorées.')
    print('     NB08 peut être utilisé en ensemble avec NB06 pour maximiser les gains.')
print()

print('  6. FICHIERS PRODUITS')
for f in [reg_path]:
    if f.exists():
        print(f'     {f.name:<35} ({f.stat().st_size/1e6:.1f} Mo)')
for t in TARGETS:
    a   = final_algo[t]
    fp  = MODELS_DIR / f'{t}_best_{a}.pkl'
    if fp.exists():
        print(f'     {fp.name:<35} ({fp.stat().st_size/1e6:.1f} Mo)')
for t in ['aqi_global', 'pm2_5']:
    fp = DATA_OUT / f'forecast_30d_{t}.csv'
    if fp.exists():
        print(f'     {fp.name:<35} ({fp.stat().st_size/1e6:.2f} Mo)')

print()
print('=' * 70)



  RAPPORT FINAL — NOTEBOOK 08 Multi-algorithmes
  IndabaX Cameroon 2026

  1. CONFIGURATION
     Train      : 2020-01-01 → 2024-09-30  (69,400 lignes)
     Validation : 2024-10-01 → 2024-12-31  (3,680 lignes)
     Test       : 2025-01-01 →   (14,160 lignes)
     Features   : 178 (base) + 5 (NO2/SO2)
     Algorithmes: LGBM, XGBoost, CatBoost + Tuning NO2/SO2 + Ensemble

  2. TABLEAU DE PERFORMANCES FINAL
  Cible                 Algo retenu      RMSE       R2    vs NB06    Objectif    OK?
  ----------------------------------------------------------------------------
  pm2_5                 stacking_bc    1.4757   0.9909      +5.4%        1.40     --
  pm10                  stacking_bc    6.4645   0.9791      +7.4%        6.50     OK
  no2                   stacking_bc    2.6536   0.7341      +5.2%        2.50     --
  o3                       stacking    8.3294   0.8957      -0.2%        7.80     --
  co                    stacking_bc   22.0061   0.9837     +11.6%       23.00     OK
  s